# 14 — Explicabilidade (SHAP) dos modelos binários (BENIGN × ataque): RF e SVM

**Pergunta do TCC:** em que atributos os modelos se apoiam para decidir, esses atributos são os mesmos no RF e no SVM, e **o que muda entre o dia do treino e o dia do teste** nas explicações (isso liga a queda de desempenho no teste ao que a validação adversarial já mostrou)? Este notebook **não treina nenhum modelo**: ele explica os modelos finais já treinados e avaliados no teste.

### O que é o SHAP, em uma frase
Para cada previsão, o SHAP divide a saída do modelo entre os atributos: cada um recebe o quanto **empurrou** a saída para cima ou para baixo em relação a uma referência, e **os valores somam exatamente a diferença até a saída** (o notebook confere isso em todas as linhas).

### O que é explicado
| | RF | SVM |
|---|---|---|
| **Explicador** | `TreeExplainer` (**exato**, sem aproximação) | `PermutationExplainer` sobre a função de decisão, com uma **referência (fundo) estratificada** do dia do treino (aproximado, mas com aditividade exata) |
| **Saída explicada** | a **probabilidade de ataque** | a **função de decisão** (a distância à fronteira; > 0 = ataque) |
| **Modelos** | os **finais (refit)**, os mesmos avaliados no teste: RF ajustado, RF padrão, SVM ajustado e SVM padrão | |

**Estratificação por tipo de ataque:** o problema é binário, mas há 12 tipos de ataque (e o `Portmap`, só no teste). A amostra tem a mesma quantidade de linhas de **cada tipo** (e de `BENIGN`), para que os tipos pequenos apareçam na explicação. A importância "geral" dá **peso igual a cada tipo**. **Fundo do SVM:** metade `BENIGN` e metade ataque (1 linha por tipo de ataque, mais o mesmo total de `BENIGN`), como o treino balanceado, do dia do treino e fora das explicadas.

**Padronização do SVM:** o SVM é sempre chamado pelo **pipeline completo** (padronização + SVM). A amostra e o fundo ficam no **espaço original** (depois da imputação, antes da padronização), e o pipeline aplica a padronização do **treino do próprio modelo** em toda previsão. O notebook **confere isso** (etapa 3.1) em vez de supor: o pipeline tem de ser igual a "padronizar e depois prever", e prever sem padronizar tem de dar outro resultado.

### Amostra estratificada (e por que ela basta)
Explicar as milhões de linhas do teste levaria dias (principalmente no SVM). O notebook explica uma **amostra estratificada por tipo de ataque**: até `N_POR_GRUPO_RF` (padrão 300) linhas de **cada** tipo de ataque para o RF e `N_POR_GRUPO_SVM` (padrão 60) para o SVM (as do SVM são **um subconjunto das do RF**, para os dois modelos serem comparados nas mesmas linhas), **em cada dia**. Grupos pequenos entram **inteiros**: nenhum fica de fora. As linhas do teste são sorteadas **entre as linhas reais** (com as repetidas), então a mistura de cada grupo é a real.

**A amostra é equivalente a usar todos os dados?** O notebook **mede isso**, em vez de supor (etapa 6.1), sempre para a importância **global**:
1. **Metades:** divide a amostra em duas metades independentes (50 sorteios) e vê se as duas dão o mesmo ranking de atributos (correlação de postos). As metades são **menores** que a amostra, então o teste é conservador.
2. **Convergência:** compara a importância com 10%, 25%, 50% e 75% da amostra com a da amostra inteira: se já com 50% o ranking é igual, o platô foi atingido.
3. **Intervalo de confiança (bootstrap):** cada atributo ganha um IC95% da sua participação (aparece nos gráficos).

**Regras de suficiência (fixadas antes de olhar os resultados):** correlação entre metades ≥ 0,90; sobreposição do top 10 entre metades ≥ 7; convergência com 50% da amostra ≥ 0,95; por tipo de ataque, correlação entre metades ≥ 0,80. Se uma regra falhar, o notebook diz **"ATENÇÃO: aumentar a amostra"** e você aumenta `N_POR_GRUPO_*`.

### O que o notebook entrega
| Etapa | O que mostra |
|---|---|
| **5.1** | os atributos que mais pesam (geral e por tipo de ataque), com a **direção** (valor alto empurra para cima ou para baixo) |
| **6.1** | **a amostra basta?** (metades, convergência, IC95%) |
| **7.1** | **dia do treino × dia do teste:** o ranking de atributos mudou? quais atributos mais mudaram de peso, com a mediana do valor em cada dia |
| **8.1** | **RF × SVM:** eles se apoiam nos mesmos atributos? |
| **9.1** | no teste, **o que separa as linhas que o modelo acerta das que ele erra** |
| **10.1** | gráficos (importância com IC95%, mapa de calor, dia × dia, convergência) e **exemplos locais** (uma linha certa e uma errada) |

### Paralelismo, progresso e retomada
- O cálculo é dividido em tarefas (pedaços de linhas), e cada tarefa roda **num processo à parte**, com **vários ao mesmo tempo só se houver memória livre** (a mesma regra do 11m e do 12m).
- **Barra de progresso** ponderada pelo tempo estimado de cada tarefa, com o tempo restante.
- Antes de calcular, o notebook **mede o tempo por linha** de cada modelo e **mostra a estimativa total**. Com `SO_ESTIMAR = True` ele só estima e para.
- Cada tarefa é salva ao terminar: **se interromper, rode de novo e ele retoma** (a pasta é reconhecida pela impressão digital da configuração).

### Garantias
**Nenhum modelo é treinado**, **nenhum arquivo das pastas dos modelos é alterado** (bloqueio por código e conferência no fim), e o servidor só é lido. Tudo é criado numa pasta nova `artefatos_modelos/shap_binario/shap_<data>`.

### O que o SHAP NÃO mostra (para escrever no TCC com honestidade)
- Ele explica **o modelo**, não a realidade: mostra o que o modelo usou, não por que o ataque é assim. **Correlação, não causa.**
- **Atributos correlacionados dividem o crédito** de forma instável (vários atributos de tamanho de pacote se parecem): não sobrevalorize a posição exata na lista.
- Os valores dependem da **referência**: no RF, a distribuição de treino das árvores; no SVM, o fundo estratificado. Por isso o RF e o SVM são comparados pela **participação (%)** e pelo ranking, não pelo valor absoluto (as unidades também diferem).
- No SVM, o fundo pequeno e as poucas permutações deixam cada explicação **ruidosa**; a importância **média** de muitas linhas é estável (etapa 6.1 mede), mas **uma linha isolada** não deve ser super-interpretada.
- No binário o SVM é pequeno (poucos milhares de vetores de suporte), então as explicações são relativamente rápidas.

### Antes de rodar
O pacote **`shap`** precisa estar instalado no seu ambiente (`pip install shap`, com a `.venv` ativa; testei a versão 0.54 com o scikit-learn 1.9.1). Precisa dos notebooks 07 e 11 do binário concluídos (os modelos finais). O RF e o SVM binários são pequenos: o notebook inteiro deve levar de **minutos a ~1 h**, e confirma a estimativa antes de começar. O `.env` precisa apontar para o servidor novo.


In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse.afonsolelis.tech:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

### Ferramentas
O pré-processamento, a leitura em CSV, as métricas, o **escalonador de tarefas condicionado à memória** e as **proteções** (sobrescrita e remoção de arquivos das pastas dos modelos bloqueadas).

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# ---------- métricas multiclasse (a partir da matriz de confusão; aceitam pesos por linha) ----------
def metricas_multi(y, proba, w=None):
    y = np.asarray(y, dtype=int); proba = np.asarray(proba, dtype=float); K_ = proba.shape[1]; pred = proba.argmax(axis=1); w_ = None if w is None else np.asarray(w, dtype=float)
    cm = np.zeros((K_, K_)); np.add.at(cm, (y, pred), 1.0 if w_ is None else w_)
    s, t, p, d = cm.sum(), cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)                  # total, linhas por classe verdadeira, previstas por classe, acertos
    rec = np.divide(d, t, out=np.full(K_, np.nan), where=t > 0); prec = np.divide(d, p, out=np.zeros(K_), where=p > 0); f1 = np.divide(2 * d, t + p, out=np.zeros(K_), where=(t + p) > 0)
    den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0)); mcc = float((d.sum() * s - (p * t).sum()) / den) if den > 0 else 0.0
    auc_k = np.full(K_, np.nan)
    for k in range(K_):
        yk = (y == k).astype(int)
        if 0 < yk.sum() < len(yk): auc_k[k] = roc_auc_score(yk, proba[:, k], sample_weight=w_)
    esc = {"acuracia": float(d.sum() / s), "acuracia_balanceada": float(np.nanmean(rec)), "precisao_macro": float(prec.mean()), "f1_macro": float(f1.mean()), "f1_ponderado": float((f1 * t).sum() / t.sum()), "mcc": mcc, "roc_auc_ovr_macro": float(np.nanmean(auc_k))}
    return esc, {"cm": cm, "recall": rec, "precisao": prec, "f1": f1, "auc": auc_k, "suporte": t}

# autoteste das ferramentas (não toca no servidor): confere as métricas multiclasse contra o scikit-learn
from sklearn.metrics import accuracy_score as _acc, balanced_accuracy_score as _bacc, f1_score as _f1, matthews_corrcoef as _mcc, precision_score as _prec
_r = np.random.default_rng(0); _y = _r.integers(0, 4, 400); _P = _r.dirichlet(np.ones(4), 400); _P[np.arange(400), _y] += 0.35; _P /= _P.sum(axis=1, keepdims=True); _w = _r.uniform(0.5, 3, 400); _pr = _P.argmax(axis=1)
for _ww in (None, _w):
    _e, _a = metricas_multi(_y, _P, _ww)
    assert abs(_e["acuracia"] - _acc(_y, _pr, sample_weight=_ww)) < 1e-12 and abs(_e["acuracia_balanceada"] - _bacc(_y, _pr, sample_weight=_ww)) < 1e-12, _e
    assert abs(_e["f1_macro"] - _f1(_y, _pr, average="macro", sample_weight=_ww)) < 1e-12 and abs(_e["f1_ponderado"] - _f1(_y, _pr, average="weighted", sample_weight=_ww)) < 1e-12, _e
    assert abs(_e["mcc"] - _mcc(_y, _pr, sample_weight=_ww)) < 1e-12 and abs(_e["precisao_macro"] - _prec(_y, _pr, average="macro", sample_weight=_ww, zero_division=0)) < 1e-12, _e
    assert abs(_e["roc_auc_ovr_macro"] - roc_auc_score(_y, _P, multi_class="ovr", average="macro", sample_weight=_ww)) < 1e-12, _e
assert abs(metricas_multi(np.array([0, 1, 2]), np.eye(3))[0]["mcc"] - 1.0) < 1e-12
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

import gc

def rss_mib():
    return psutil.Process().memory_info().rss / 2 ** 20 if psutil else float("nan")

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")


def recall_por_tipo(y, p, g, limiar, n_tipos):
    # Posição 0 = BENIGN (aqui vale a especificidade); posições seguintes = recall de cada tipo de ataque.
    y, g = np.asarray(y), np.asarray(g); pred = np.asarray(p) >= np.float32(limiar)
    pos = np.bincount(g[y == 1], minlength=n_tipos).astype(float); det = np.bincount(g[(y == 1) & pred], minlength=n_tipos).astype(float)
    r = np.divide(det, pos, out=np.full(n_tipos, np.nan), where=pos > 0); neg = float((y == 0).sum()); r[0] = float(((y == 0) & ~pred).sum()) / neg if neg else float("nan")
    return r

class ErroMemoriaServidor(Exception):
    pass


CH_SETTINGS_LEITURA = globals().get("CH_SETTINGS_LEITURA", {"max_threads": 1, "max_block_size": 8192})      # leitura leve para o servidor (menos colunas e blocos abertos ao mesmo tempo)
ESPERAS_MEMORIA = globals().get("ESPERAS_MEMORIA", (30, 60, 120, 240, 480))                            # segundos de espera após "sem memória" (Code 241) no servidor
LIMITE_MEM_SERVIDOR_GIB = globals().get("LIMITE_MEM_SERVIDOR_GIB", 2.0)                                        # volta a tentar antes do fim da espera se a memória do servidor cair abaixo disto


def ler_csv(query, colunas, texto=()):
    # Leitura em FLUXO: o servidor devolve CSV e o pandas lê direto da conexão, sem guardar o texto inteiro nem criar milhões de objetos Python
    # (o caminho JSON do consultar() gasta várias vezes mais memória). Mesmas regras de repetição do _http: só SELECT, com novas tentativas.
    global _conn
    if not re.match(r"\s*SELECT\b", query, re.I): raise PermissionError("ler_csv só executa SELECT")
    caminho = "/?" + urllib.parse.urlencode({"database": CH_DATABASE, "default_format": "CSV", "wait_end_of_query": 1, **CH_SETTINGS_LEITURA})
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    ultimo = None
    for espera in CONEXAO_ESPERAS:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s..."); time.sleep(espera)
        corpo = b""
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            if resp.status == 200:
                try:
                    return pd.read_csv(resp, header=None, names=colunas, dtype={c: (str if c in texto else np.float64) for c in colunas}, na_values=["\\N"])
                except pd.errors.EmptyDataError:
                    return pd.DataFrame({c: pd.Series(dtype=(object if c in texto else np.float64)) for c in colunas})
            corpo = resp.read()
        except (OSError, http.client.HTTPException, pd.errors.ParserError, ValueError) as e:
            ultimo = e; _fechar_conexao(); continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} corpo={corpo[:300].decode('utf-8', 'replace')!r}"); _fechar_conexao(); continue
        msg = corpo.decode("utf-8", "replace")
        if "MEMORY_LIMIT_EXCEEDED" in msg or msg.startswith("Code: 241"):
            raise ErroMemoriaServidor(msg[:600])
        raise Exception(msg[:2000])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(CONEXAO_ESPERAS)} tentativas: {ultimo!r}")


def _mem_servidor_gib():
    try:
        return float(consultar_num("SELECT toFloat64(value) AS v FROM system.asynchronous_metrics WHERE metric = 'MemoryResident'")["v"][0]) / 2 ** 30
    except Exception:
        return None


def ler_csv_com_espera(query, colunas, texto=()):
    # Se o servidor ficar sem memória (Code 241, comum num servidor de 3 GiB com merges em andamento), espera e tenta de novo; o erro é transitório.
    for k, espera in enumerate(list(ESPERAS_MEMORIA) + [None]):
        try:
            return ler_csv(query, colunas, texto)
        except ErroMemoriaServidor as e:
            if espera is None:
                raise ErroMemoriaServidor("O servidor continua sem memória depois de todas as esperas. Rode a célula de novo mais tarde: os lotes já prontos são reaproveitados. Último erro: " + str(e)[:300]) from None
            m = _mem_servidor_gib()
            print(f"  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: {'?' if m is None else f'{m:.2f} GiB'}); esperando até {espera}s e tentando de novo ({k + 1}/{len(ESPERAS_MEMORIA)})...", flush=True)
            t0 = time.time()
            while time.time() - t0 < espera:
                time.sleep(min(15, espera))
                m = _mem_servidor_gib()
                if m is not None and m < LIMITE_MEM_SERVIDOR_GIB: break

_b = Barra(2, "autoteste"); _b.avancar(2); assert recall_por_tipo(np.array([0, 1, 1]), np.array([.1, .9, .2]), np.array([0, 1, 1]), 0.5, 2)[1] == 0.5
print("Autoteste das ferramentas do teste OK")

# ---------- métricas multiclasse para arrays grandes (sem cópias em float64; confusão por contagem) ----------
def metricas_multi_grande(y, P):
    y = np.asarray(y, dtype=np.int64); K_ = P.shape[1]; pred = P.argmax(axis=1)
    cm = np.bincount(y * K_ + pred, minlength=K_ * K_).reshape(K_, K_).astype(np.float64)
    s, t, p, d = cm.sum(), cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)
    rec = np.divide(d, t, out=np.full(K_, np.nan), where=t > 0); prec = np.divide(d, p, out=np.zeros(K_), where=p > 0); f1 = np.divide(2 * d, t + p, out=np.zeros(K_), where=(t + p) > 0)
    den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0)); mcc = float((d.sum() * s - (p * t).sum()) / den) if den > 0 else 0.0
    auc_k = np.full(K_, np.nan)
    for k in range(K_):
        yk = (y == k)
        if 0 < yk.sum() < len(yk): auc_k[k] = roc_auc_score(yk, np.ascontiguousarray(P[:, k], dtype=np.float64))
    esc = {"acuracia": float(d.sum() / s), "acuracia_balanceada": float(np.nanmean(rec)), "precisao_macro": float(prec.mean()), "f1_macro": float(f1.mean()), "f1_ponderado": float((f1 * t).sum() / t.sum()), "mcc": mcc, "roc_auc_ovr_macro": float(np.nanmean(auc_k))}
    return esc, {"cm": cm, "recall": rec, "precisao": prec, "f1": f1, "auc": auc_k, "suporte": t, "previstas": p}

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = k / n; c = (ph + z * z / (2 * n)) / (1 + z * z / n); h = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / (1 + z * z / n); return (c - h, c + h)

# autoteste (não toca no servidor): confere contra o scikit-learn
from sklearn.metrics import accuracy_score as _acc, balanced_accuracy_score as _bacc, f1_score as _f1, matthews_corrcoef as _mcc
_r = np.random.default_rng(1); _y = _r.integers(0, 4, 500); _P = _r.dirichlet(np.ones(4), 500); _P[np.arange(500), _y] += 0.3; _P = (_P / _P.sum(axis=1, keepdims=True)).astype(np.float32); _pr = _P.argmax(axis=1)
_e, _a = metricas_multi_grande(_y, _P)
assert abs(_e["acuracia"] - _acc(_y, _pr)) < 1e-12 and abs(_e["acuracia_balanceada"] - _bacc(_y, _pr)) < 1e-12 and abs(_e["f1_macro"] - _f1(_y, _pr, average="macro")) < 1e-12 and abs(_e["mcc"] - _mcc(_y, _pr)) < 1e-12, _e
assert abs(_e["roc_auc_ovr_macro"] - roc_auc_score(_y, _P.astype(np.float64), multi_class="ovr", average="macro")) < 1e-9
_lo, _hi = wilson(50, 100); assert 0.40 < _lo < 0.41 and 0.59 < _hi < 0.60
print("Autoteste das ferramentas do teste multiclasse OK")

# ---------- escalonador de tarefas condicionado à memória (o mesmo do 11m e do 12m) ----------
import collections
try:
    import shap
except ImportError as e:
    raise ImportError("O pacote shap não está instalado. Com a .venv ativa, rode:  pip install shap   (e reinicie o kernel)") from e
from joblib.externals.loky import ProcessPoolExecutor as LokyPool
import concurrent.futures
def ram_livre_gib():
    s = globals().get("RAM_LIVRE_SIMULADA")                                                                     # só para testes
    return float(s() if callable(s) else s) if s is not None else (psutil.virtual_memory().available / 2 ** 30 if psutil else float("inf"))

def processos_seguros(desejado, ram_por_proc_gib):
    # Quantos processos simultâneos cabem AGORA: cada um precisa de ram_por_proc × 1,3, e a reserva nunca é tocada. Sempre devolve pelo menos 1 (nunca trava), mas avisa se nem 1 cabe folgado.
    livre = ram_livre_gib() - RAM_RESERVA_GIB
    if not np.isfinite(livre): return int(desejado)
    return int(max(1, min(desejado, livre // (ram_por_proc_gib * 1.3))))

def rodar_tarefas(pendentes, lancar, coletar, barra, limite_s, k_max, ram_proc, rotulo, parar_apos=None):
    # Escalonador de treinos independentes, UM PROCESSO NOVO por treino. Só inicia mais um treino se houver memória livre para ele (ram_proc × 1,3 + reserva) e se o anterior já ocupou a sua memória;
    # com um treino só rodando, sempre inicia o próximo (nunca trava). Se a memória ficar crítica com vários treinos, o mais novo é interrompido e volta para a fila. Treino acima do limite de tempo é encerrado.
    espera_desde = None; fila, rodando, feitos, ultima, stats = collections.deque(pendentes), [], 0, 0.0, {"max_simultaneos": 0, "interrompidos_por_memoria": 0, "ram_proc_final": ram_proc}
    try:
        while fila or rodando:
            while fila and len(rodando) < k_max:
                livre = ram_livre_gib(); agora = time.perf_counter()
                if rodando and (agora - ultima < INTERVALO_PARTIDAS_S or livre - RAM_RESERVA_GIB < ram_proc * 1.3): break
                if not rodando:
                    if livre < RAM_MINIMA_UM_TREINO_GIB:                                  # com nada rodando, espera a memória liberar (até 10 min); depois, para com uma mensagem clara
                        espera_desde = espera_desde or agora
                        if agora - espera_desde > 600: raise RuntimeError(f"Falta memória livre: um treino precisa de pelo menos {RAM_MINIMA_UM_TREINO_GIB:.1f} GiB livres e há {livre:.1f} GiB. Feche o Meet, o Edge e os outros notebooks (ou defina RAM_MINIMA_UM_TREINO_GIB mais baixo, por sua conta e risco).")
                        barra._extra = f"aguardando memória livre ({livre:.1f} GiB; preciso de pelo menos {RAM_MINIMA_UM_TREINO_GIB:.1f} GiB)..."; barra._mostrar(); time.sleep(2.0); break
                    espera_desde = None
                t = fila.popleft(); ex, fut = lancar(t); rodando.append({"t": t, "ex": ex, "fut": fut, "t0": time.perf_counter(), "k": len(rodando) + 1}); ultima = time.perf_counter(); stats["max_simultaneos"] = max(stats["max_simultaneos"], len(rodando))
            for r in list(rodando):
                dec = time.perf_counter() - r["t0"]
                if r["fut"].done():
                    res = r["fut"].result(); r["ex"].shutdown(wait=True); rodando.remove(r); feitos += 1; ram_proc = max(ram_proc, 1.15 * res[0]["pico_ram_mib"] / 1024); stats["ram_proc_final"] = ram_proc; coletar(r["t"], res, r["k"])
                    if parar_apos and feitos >= parar_apos: raise RuntimeError("parada de teste (PARAR_APOS_FITS)")
                elif dec > limite_s: r["ex"].shutdown(wait=True, kill_workers=True); rodando.remove(r); feitos += 1; coletar(r["t"], None, r["k"])
            if len(rodando) > 1 and ram_livre_gib() < RAM_CRITICA_GIB:
                r = rodando.pop(); r["ex"].shutdown(wait=True, kill_workers=True); fila.appendleft(r["t"]); k_max = max(1, k_max - 1); stats["interrompidos_por_memoria"] += 1
                print(f"  ⚠ memória livre crítica ({ram_livre_gib():.2f} GiB): um treino foi interrompido e volta para a fila; o limite de treinos simultâneos agora é {k_max}")
            barra._extra = f"{rotulo} | {len(rodando)} treino(s) rodando (até {k_max}) | memória livre {ram_livre_gib():.1f} GiB | " + (f"mais antigo há {fmt_tempo(max(time.perf_counter() - r['t0'] for r in rodando))}" if rodando else "")
            barra._mostrar(); time.sleep(0.5)
    except BaseException:
        for r in rodando: r["ex"].shutdown(wait=True, kill_workers=True)
        raise
    return stats

# ---------- proteções: o que este notebook NUNCA faz (sobrescrever ou apagar arquivos das pastas dos modelos) ----------
import warnings, shutil
def silenciar_aviso_sklearn():
    # Com muitos núcleos, o scikit-learn às vezes deixa a lista de filtros de aviso VAZIA (condição de corrida entre threads) e passa a emitir, a cada árvore, um aviso inofensivo
    # sobre `delayed`/`Parallel`. Este desvio descarta SÓ essa mensagem; os outros avisos continuam funcionando. Não muda nenhum resultado.
    warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*", category=UserWarning)
    if not getattr(warnings.warn, "_filtrado_sklearn", False):
        _orig = warnings.warn
        def _warn_filtrado(message, *args, **kwargs):
            if isinstance(message, str) and message.startswith("`sklearn.utils.parallel.delayed` should be used with"): return
            return _orig(message, *args, **kwargs)
        _warn_filtrado._filtrado_sklearn = True; warnings.warn = _warn_filtrado
silenciar_aviso_sklearn()

RAIZES_TRAVADAS, CRIADOS, MANTIDOS = [], [], []
def instalar_travas(pasta, permitir_regravar=()):
    # Bloqueia gravação que SOBRESCREVA arquivo existente dentro das pastas protegidas (as dos modelos), e qualquer remoção de arquivo dentro delas. O treino NÃO é bloqueado (o diagnóstico treina modelos de teste).
    # Pode ser chamada para várias pastas; só arquivos temporários (.tmp) e acréscimos (modo "a") podem ser regravados.
    from sklearn.ensemble import RandomForestClassifier as _RF
    from sklearn.svm import SVC as _SVC
    import numpy as _np, pandas as _pd, joblib as _jl, pathlib as _pl
    RAIZES_TRAVADAS.append(str(Path(pasta).resolve())); perm = set(permitir_regravar)
    if getattr(os.remove, "_travado", False): return
    def dentro(alvo):
        try: s_ = str(Path(alvo).resolve())
        except Exception: return False
        return any(s_.startswith(r) for r in RAIZES_TRAVADAS)
    def checa(alvo, acao, permitir_acrescimo=False):
        if not isinstance(alvo, (str, os.PathLike)) or not dentro(alvo): return
        p = Path(alvo).resolve()
        if p.exists() and p.name not in perm and ".tmp" not in p.name and not permitir_acrescimo:
            raise PermissionError(f"BLOQUEADO: {acao} sobrescreveria '{p.name}', que já existe. Este notebook só cria arquivos novos.")
    def embrulha(obj, nome, pos, acao, kw="", acrescimo_ok=False):
        orig = getattr(obj, nome)
        if getattr(orig, "_travado", False): return
        def f(*a, **k):
            alvo = a[pos] if len(a) > pos else k.get(kw)
            checa(alvo, acao, permitir_acrescimo=(acrescimo_ok and k.get("mode") == "a")); return orig(*a, **k)
        f._travado = True; f.__name__ = getattr(orig, "__name__", nome); setattr(obj, nome, f)
    embrulha(_pd.DataFrame, "to_csv", 1, "gravar CSV", "path_or_buf", acrescimo_ok=True); embrulha(_np, "savez", 0, "gravar NPZ", "file"); embrulha(_np, "savez_compressed", 0, "gravar NPZ", "file"); embrulha(_np, "save", 0, "gravar NPY", "file")
    embrulha(_jl, "dump", 1, "gravar modelo", "filename"); embrulha(_pl.Path, "write_text", 0, "gravar arquivo"); embrulha(_pl.Path, "write_bytes", 0, "gravar arquivo")
    embrulha(os, "replace", 1, "substituir arquivo"); embrulha(os, "rename", 1, "renomear arquivo")
    try:
        from matplotlib.figure import Figure; embrulha(Figure, "savefig", 1, "gravar figura", "fname")
    except ImportError: pass
    def _proibe_remocao(orig):
        def f(*a, **k):
            alvo = a[0] if a else None
            if isinstance(alvo, (str, os.PathLike)) and dentro(alvo): raise PermissionError(f"BLOQUEADO: apagar '{Path(alvo).name}'. Este notebook nunca apaga arquivos.")
            return orig(*a, **k)
        f._travado = True; return f
    if not getattr(os.remove, "_travado", False): os.remove = _proibe_remocao(os.remove)
    if not getattr(_pl.Path.unlink, "_travado", False):
        _unl = _pl.Path.unlink
        def _u(self, *a, **k):
            if dentro(self): raise PermissionError(f"BLOQUEADO: apagar '{self.name}'. Este notebook nunca apaga arquivos.")
            return _unl(self, *a, **k)
        _u._travado = True; _pl.Path.unlink = _u

def gravar(caminho, escrever):
    # Cria o arquivo SÓ se ele ainda NÃO existir (nunca sobrescreve). Registra o que foi criado e o que foi mantido.
    caminho = Path(caminho); rotulo = f"{caminho.parent.name}/{caminho.name}"
    if caminho.exists(): MANTIDOS.append(rotulo); return False
    caminho.parent.mkdir(parents=True, exist_ok=True); escrever(caminho); CRIADOS.append(rotulo); return True
def gravar_csv(df, caminho, **kw): return gravar(caminho, lambda p: df.to_csv(p, **kw))
def gravar_json(obj, caminho): return gravar(caminho, lambda p: salvar_json(obj, p))
def tmp_novo(caminho): return Path(caminho).with_name(Path(caminho).stem + ".tmp" + Path(caminho).suffix)

print("Ferramentas do SHAP prontas | shap", shap.__version__)


Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2


Autoteste das ferramentas do teste OK
Autoteste das ferramentas do teste multiclasse OK
Ferramentas do SHAP prontas | shap 0.54.1


## 1. Parâmetros, pastas e modelos
Localiza os modelos finais (RF e SVM), **ativa as proteções** e define o tamanho das amostras. Tudo pode ser trocado definindo a variável antes de rodar a célula (por exemplo `N_POR_GRUPO_RF = 300`).

In [6]:
INCLUIR_PADRAO_RF_PADRAO, INCLUIR_PADRAO_SVM_PADRAO = True, True                  # no binário os modelos são pequenos: os quatro entram por padrão
N_POR_GRUPO_RF_PADRAO, N_POR_GRUPO_SVM_PADRAO = 300, 60                            # linhas por tipo de ataque e por dia
N_BG_POR_GRUPO_SVM_PADRAO, PERMUTACOES_SVM_PADRAO = 1, 1                           # fundo do SVM: 1 linha por tipo; 1 permutação (direta e inversa) por linha (use 2 para menos ruído)
CHUNK_RF_PADRAO, CHUNK_SVM_PADRAO = 1000, 40

# ---------- 1.1 Parâmetros comuns ----------
import joblib, hashlib, collections, math, shutil, zlib
from scipy.stats import rankdata
import shap
REPO = Path(globals().get("REPO_ROOT", Path.cwd())); BASE = REPO / "artefatos_modelos"
SEMENTE = globals().get("SEMENTE", 42)
DIAS = ("treino", "teste"); ROT_DIA = {"treino": "dia do treino", "teste": "dia do teste"}                  # "dia do treino" = linhas que o modelo viu no treino (refit); "dia do teste" = o outro dia
INCLUIR_PADRAO_RF = globals().get("INCLUIR_PADRAO_RF", INCLUIR_PADRAO_RF_PADRAO); INCLUIR_PADRAO_SVM = globals().get("INCLUIR_PADRAO_SVM", INCLUIR_PADRAO_SVM_PADRAO)
N_POR_GRUPO_RF = globals().get("N_POR_GRUPO_RF", N_POR_GRUPO_RF_PADRAO); N_POR_GRUPO_SVM = globals().get("N_POR_GRUPO_SVM", N_POR_GRUPO_SVM_PADRAO)       # linhas explicadas por grupo (classe ou tipo de ataque) e por dia; grupos menores entram INTEIROS
N_BG_POR_GRUPO_SVM = globals().get("N_BG_POR_GRUPO_SVM", N_BG_POR_GRUPO_SVM_PADRAO)       # linhas de referência (fundo) do SVM por grupo: o custo do SVM é proporcional ao tamanho do fundo
PERMUTACOES_SVM = globals().get("PERMUTACOES_SVM", PERMUTACOES_SVM_PADRAO)                # permutações por linha no SVM (1 = direta e inversa; mais = menos ruído, custo proporcional)
CHUNK_RF, CHUNK_SVM = globals().get("CHUNK_RF", CHUNK_RF_PADRAO), globals().get("CHUNK_SVM", CHUNK_SVM_PADRAO)         # linhas por tarefa (cada tarefa é salva ao terminar: retomável)
SO_ESTIMAR = globals().get("SO_ESTIMAR", False)                           # True = mede o tempo por linha, mostra a estimativa e PARA antes de calcular
NOVA_EXECUCAO = globals().get("NOVA_EXECUCAO", False)                     # True = não retomar uma execução interrompida
N_SPLITS_ESTAB, N_BOOT, TOP_K = globals().get("N_SPLITS_ESTAB", 50), globals().get("N_BOOT", 200), globals().get("TOP_K", 10)
# --- paralelismo condicionado à memória (a mesma regra do 11m e do 12m) ---
PROCESSOS_MAX = globals().get("PROCESSOS_MAX", 4); RAM_POR_PROCESSO_GIB = globals().get("RAM_POR_PROCESSO_GIB", 1.0)
RAM_RESERVA_GIB = globals().get("RAM_RESERVA_GIB", 1.0); RAM_MINIMA_UM_TREINO_GIB = globals().get("RAM_MINIMA_UM_TREINO_GIB", 0.5); RAM_CRITICA_GIB = globals().get("RAM_CRITICA_GIB", 0.4)
INTERVALO_PARTIDAS_S = globals().get("INTERVALO_PARTIDAS_S", 10); LIMITE_TAREFA_S = globals().get("LIMITE_TAREFA_S", 6 * 3600); PARAR_APOS_TAREFAS = globals().get("PARAR_APOS_TAREFAS", None)       # só para testes
# regras de suficiência da amostra (fixadas ANTES de olhar os resultados)
REGRAS_ESTAB = {"rho_metade_geral": 0.90, "sobreposicao_top10_geral": 7, "rho_convergencia_50pct": 0.95, "rho_metade_grupo": 0.80}
T_SEC = {}
def foto(raiz, rot): return {(rot + "/" + str(f.relative_to(raiz))): (f.stat().st_size, f.stat().st_mtime_ns) for f in Path(raiz).rglob("*") if f.is_file()}
def ultima(base, padrao, exigir):
    ps = sorted(p for p in (BASE / base).glob(padrao) if all((p / f).exists() for f in exigir)); return ps[-1] if ps else None

MODO, GRUPO_ROTULO = "bin", "tipo de ataque"
EXEMPLOS = globals().get("EXEMPLOS", ["Syn", "UDPLag", "BENIGN"])                    # tipos mostrados nos exemplos locais (uma linha certa e uma errada de cada)
PASTA_RAIZ = BASE / "shap_binario"; PASTA_RAIZ.mkdir(exist_ok=True)
PASTA_RF = Path(globals().get("PASTA_RF") or ultima("rf_binario_busca", "rf_busca_*", ("modelo_final_refit.joblib", "FINALIZADO")) or ".")
PASTA_RF_PAD = Path(globals().get("PASTA_RF_PAD") or ultima("rf_binario", "rf_bin_*", ("modelo_final_refit.joblib",)) or ".")
PASTA_SVM = Path(globals().get("PASTA_SVM") or ultima("svm_binario_busca", "svm_busca_*", ("modelo_final_refit.joblib", "FINALIZADO")) or ".")
assert (PASTA_RF / "modelo_final_refit.joblib").exists() and (PASTA_SVM / "modelo_final_refit.joblib").exists(), "faltam os modelos finais (notebooks 07 e 11 do binário concluídos)"
V_REFIT, SEL_TAB, VIEW_TESTE, TAB_TESTE, AUD_TAB = "prep_vw_bal_bin_refit", "prep_bal_bin_selecao", "prep_vw_rf_bin_teste_sem_sobrep", "prep_final_teste", "prep_views_rf_binario"
faltam = [t for t in (V_REFIT, SEL_TAB, VIEW_TESTE, TAB_TESTE, AUD_TAB) if not existe(t)]; assert not faltam, f"Rode antes os notebooks 02, 03 e 04 do binário: faltam {faltam}"
PASTAS_MODELOS = {"svm": PASTA_SVM, "rf_aj": PASTA_RF, **({"rf_pad": PASTA_RF_PAD} if INCLUIR_PADRAO_RF else {})}
SNAP_ANTES = {n: v for r_, p_ in PASTAS_MODELOS.items() for n, v in foto(p_, r_).items()}
for p_ in PASTAS_MODELOS.values(): instalar_travas(p_)
silenciar_aviso_sklearn()
aud = consultar_num(f"SELECT posicao, feature, situacao, mediana_imputacao, tem_indicador, versao FROM {AUD_TAB} ORDER BY posicao"); versao_views = str(aud["versao"].iloc[0])
USADAS, RETIRADAS = aud[aud.situacao == "usada"], aud[aud.situacao != "usada"].feature.tolist(); MEDIANAS = {r.feature: float(r.mediana_imputacao) for r in USADAS.itertuples() if int(r.tem_indicador) == 1}
prep = PreprocessadorRF(USADAS.feature.tolist(), MEDIANAS, RETIRADAS, versao_views); CM = prep.colunas_modelo; D = len(CM)
assert [c for c in colunas_view(V_REFIT) if c not in ("id_linha", "y")] == CM == [c for c in colunas_view(VIEW_TESTE) if c not in ("id_linha", "y")], "a ordem das colunas das views difere"
VERSOES = {"views": versao_views, "selecao": str(consultar(f"SELECT any(versao) AS v FROM {SEL_TAB}")["v"][0])}
cfg_s = json.loads((PASTA_SVM / "config_treino.json").read_text(encoding="utf-8")) if (PASTA_SVM / "config_treino.json").exists() else {}
LIM_SVM = float(cfg_s.get("limiar_padrao", 0.0))
MODELOS = {}
def registrar(chave, rotulo, familia, arquivo, descricao):
    m = joblib.load(arquivo)
    if familia == "RF": assert list(getattr(m, "colunas_modelo_", CM)) == CM, f"{rotulo}: as colunas do modelo diferem das das views"
    lim = 0.5 if familia == "RF" else LIM_SVM
    MODELOS[chave] = dict(rotulo=rotulo, familia=familia, arquivo=Path(arquivo), modelo=m, rotular=(lambda FX, lim=lim: (FX[:, 0] >= lim).astype(int)), descricao=descricao)
hp_rf = (json.loads((PASTA_RF / "config_treino.json").read_text(encoding="utf-8")).get("hiperparametros", {}) if (PASTA_RF / "config_treino.json").exists() else {}); hp_sv = cfg_s.get("hiperparametros", {})
registrar("rf_ajustado", "RF ajustado", "RF", PASTA_RF / "modelo_final_refit.joblib", f"{hp_rf.get('n_estimators')} árvores | profundidade {hp_rf.get('max_depth')} | folha {hp_rf.get('min_samples_leaf')} | {PASTA_RF.name}")
if INCLUIR_PADRAO_RF: registrar("rf_padrao", "RF padrão", "RF", PASTA_RF_PAD / "modelo_final_refit.joblib", f"100 árvores | profundidade livre | folha 1 | {PASTA_RF_PAD.name}")
registrar("svm_ajustado", "SVM ajustado", "SVM", PASTA_SVM / "modelo_final_refit.joblib", f"C = {hp_sv.get('C')} | gamma = {hp_sv.get('gamma_fator', hp_sv.get('gamma'))}× | {PASTA_SVM.name}")
if INCLUIR_PADRAO_SVM and (PASTA_SVM / "modelo_padrao_refit.joblib").exists(): registrar("svm_padrao", "SVM padrão", "SVM", PASTA_SVM / "modelo_padrao_refit.joblib", f"C = 1 | gamma = 1× | {PASTA_SVM.name}")
elif INCLUIR_PADRAO_SVM: print("⚠ o SVM padrão (modelo_padrao_refit.joblib) não foi encontrado: fica de fora")
print(f"Binário (BENIGN × ataque) | {D} atributos"); print("\nModelos finais (treinados no refit; os mesmos avaliados no teste):")
for k, M in MODELOS.items(): print(f"  {M['rotulo']:13s} limiar natural {0.5 if M['familia'] == 'RF' else LIM_SVM:g} | {M['descricao']}")
print(f"\n🔒 Proteções: sobrescrita e remoção BLOQUEADAS nas pastas dos modelos ({len(SNAP_ANTES)} arquivos já existem e serão conferidos no final); o servidor só é lido.")


Binário (BENIGN × ataque) | 43 atributos

Modelos finais (treinados no refit; os mesmos avaliados no teste):
  RF ajustado   limiar natural 0.5 | 300 árvores | profundidade 15 | folha 1 | rf_busca_20261024_175105
  RF padrão     limiar natural 0.5 | 100 árvores | profundidade livre | folha 1 | rf_bin_20261021_221311
  SVM ajustado  limiar natural 0 | C = 100.0 | gamma = 4.0× | svm_busca_20261025_004828
  SVM padrão    limiar natural 0 | C = 1 | gamma = 1× | svm_busca_20261025_004828

🔒 Proteções: sobrescrita e remoção BLOQUEADAS nas pastas dos modelos (778 arquivos já existem e serão conferidos no final); o servidor só é lido.


## 2. Amostras estratificadas: dia do treino e dia do teste
Sorteia, **em cada dia**, até N linhas de **cada tipo de ataque** (e de `BENIGN`; tipos pequenos entram inteiros). O dia do treino vem das linhas do refit; o do teste, de uma consulta ao servidor (só `SELECT`). Mostra quantas linhas de cada tipo foram explicadas.

In [7]:
# ---------- amostra estratificada (funções) ----------
def amostra_estratificada(grupo, n_por_grupo, semente):
    # Sorteia até n_por_grupo linhas de CADA grupo (classe ou tipo de ataque). Grupos com menos linhas entram INTEIROS: nenhum grupo minoritário fica de fora.
    # Devolve os índices (em ordem crescente) e a posição de cada linha na ordem aleatória do seu grupo (a "ordem de sorteio"): as primeiras n linhas de cada grupo formam uma subamostra estratificada.
    rng = np.random.default_rng(semente); idx, rk = [], []
    for g in np.unique(grupo):
        pos = np.flatnonzero(grupo == g); sel = rng.permutation(pos)[:n_por_grupo]; idx.append(sel); rk.append(np.arange(len(sel)))
    idx, rk = np.concatenate(idx), np.concatenate(rk); o = np.argsort(idx, kind="stable"); return idx[o], rk[o]
def construir_amk():
    # A mesma amostra principal serve aos dois modelos: o RF usa as primeiras N_POR_GRUPO_RF linhas de cada grupo e o SVM, as primeiras N_POR_GRUPO_SVM (subamostra da do RF).
    out = {}
    for k, M in MODELOS.items():
        n = N_POR_GRUPO_RF if M["familia"] == "RF" else N_POR_GRUPO_SVM; out[k] = {}
        for d in DIAS:
            A = AMOSTRA[d]; m = A["rank"] < n; out[k][d] = {**{c: A[c][m] for c in ("X", "y", "grupo", "id")}, "idx": np.flatnonzero(m)}
    return out
def tab_amostra(dia, n_rf, n_svm):
    A = AMOSTRA[dia]; return pd.DataFrame({"linhas disponíveis": [A["disp"].get(g, 0) for g in GRUPOS_DIA[dia]], "explicadas pelo RF": [int(((A["grupo"] == i) & (A["rank"] < n_rf)).sum()) for i in range(len(GRUPOS_DIA[dia]))],
                                          "explicadas pelo SVM": [int(((A["grupo"] == i) & (A["rank"] < n_svm)).sum()) for i in range(len(GRUPOS_DIA[dia]))]}, index=GRUPOS_DIA[dia])

# ---------- 2.1 Amostras estratificadas por tipo de ataque: dia do treino e dia do teste ----------
t_sec0 = time.perf_counter()
def ordenar(df): return df.iloc[np.argsort(df["id_linha"].astype("uint64").to_numpy(), kind="stable")].reset_index(drop=True)
nome_tipo = lambda t: re.sub(r"^DrDoS_", "", str(t))
rf_df = ordenar(ler_view(V_REFIT)); X_rf, y_rf, id_rf = rf_df[CM].to_numpy(float), rf_df["y"].to_numpy(int), rf_df["id_linha"].astype("uint64").to_numpy(); assert np.isfinite(X_rf).all(), "valores não finitos no refit"
sel = consultar(f"SELECT toString(id_linha) AS id_linha, tipo FROM {SEL_TAB}"); sel["tipo"] = sel["tipo"].map(nome_tipo)
tipo_rf = rf_df["id_linha"].astype("uint64").astype(str).map(dict(zip(sel["id_linha"], sel["tipo"]))); assert tipo_rf.notna().all(), "há linhas do refit sem tipo de ataque na tabela de seleção"
TIPOS_TR = ["BENIGN"] + sorted(t for t in set(tipo_rf) if t != "BENIGN"); g_rf = tipo_rf.map({t: i for i, t in enumerate(TIPOS_TR)}).to_numpy(int); assert ((g_rf == 0) == (y_rf == 0)).all(), "o tipo BENIGN não coincide com y = 0"
N_MAX = max(N_POR_GRUPO_RF if any(M["familia"] == "RF" for M in MODELOS.values()) else 0, N_POR_GRUPO_SVM if any(M["familia"] == "SVM" for M in MODELOS.values()) else 0)
AMOSTRA = {}
idx_tr, rk_tr = amostra_estratificada(g_rf, N_MAX, SEMENTE)
AMOSTRA["treino"] = {"X": X_rf[idx_tr], "y": y_rf[idx_tr], "grupo": g_rf[idx_tr], "id": id_rf[idx_tr], "rank": rk_tr, "disp": {t: int((g_rf == i).sum()) for i, t in enumerate(TIPOS_TR)}}
# dia do teste: amostra por tipo no servidor (as linhas reais, repetidas incluídas: a mistura de cada tipo é a real); só o SELECT
cont = consultar_num(f"SELECT label_multi AS tipo, count() AS n FROM {TAB_TESTE} WHERE sobrepoe_treino = 0 GROUP BY label_multi"); cont["tipo"] = cont["tipo"].map(nome_tipo); cont = cont.groupby("tipo", as_index=False)["n"].sum(); N_TESTE_TIPO = {r.tipo: int(r.n) for r in cont.itertuples()}
TIPOS_TE = ["BENIGN"] + sorted(t for t in N_TESTE_TIPO if t != "BENIGN"); cols_sql = ", ".join("v." + q(c) for c in CM)
X_te, y_te, g_te, id_te, rk_te = [], [], [], [], []
barra = Barra(len(TIPOS_TE), "Lendo a amostra do teste, por tipo")
for gi, t in enumerate(TIPOS_TE):
    ordem_ids = consultar(f"SELECT toString(id_linha) AS id_linha FROM {TAB_TESTE} WHERE label_multi IN ({lit(t)}, {lit('DrDoS_' + t)}) AND sobrepoe_treino = 0 ORDER BY cityHash64(id_linha, {int(SEMENTE)}) LIMIT {int(N_MAX)}")["id_linha"].tolist()
    ids_sql = ", ".join(ordem_ids); df = ler_csv_com_espera(f"SELECT toString(v.id_linha) AS id, v.y, {cols_sql} FROM {VIEW_TESTE} AS v WHERE v.id_linha IN ({ids_sql})", ["id", "y"] + CM, texto=("id",))
    pos = {i_: r for r, i_ in enumerate(ordem_ids)}; df["rk"] = df["id"].map(pos); df = df.sort_values("rk").reset_index(drop=True); assert len(df) == len(ordem_ids), f"{t}: a view devolveu {len(df)} de {len(ordem_ids)} linhas"
    X_te.append(df[CM].to_numpy(float)); y_te.append(df["y"].to_numpy(int)); g_te.append(np.full(len(df), gi)); id_te.append(df["id"].astype("uint64").to_numpy()); rk_te.append(df["rk"].to_numpy(int)); barra.avancar(1, f"{t}: {len(df):,} linhas de {N_TESTE_TIPO[t]:,}")
barra.fechar()
X_te, y_te, g_te, id_te, rk_te = np.vstack(X_te), np.concatenate(y_te), np.concatenate(g_te), np.concatenate(id_te), np.concatenate(rk_te); o = np.argsort(id_te, kind="stable"); X_te, y_te, g_te, id_te, rk_te = X_te[o], y_te[o], g_te[o], id_te[o], rk_te[o]
assert np.isfinite(X_te).all()
AMOSTRA["teste"] = {"X": X_te, "y": y_te, "grupo": g_te, "id": id_te, "rank": rk_te, "disp": N_TESTE_TIPO}; GRUPOS_DIA = {"treino": TIPOS_TR, "teste": TIPOS_TE}
# fundo (referência) do SVM: linhas do dia do treino FORA das explicadas, com a MESMA proporção do treino balanceado: metade BENIGN e metade ataque (N_BG_POR_GRUPO_SVM por tipo de ataque; BENIGN = o mesmo total de linhas de ataque)
resto = np.setdiff1d(np.arange(len(y_rf)), idx_tr); g_resto = g_rf[resto]; rng_bg = np.random.default_rng(SEMENTE + 2)
ib_atq = np.concatenate([rng_bg.permutation(np.flatnonzero(g_resto == gi))[:N_BG_POR_GRUPO_SVM] for gi in range(1, len(TIPOS_TR))]); N_BG_BENIGN = int(globals().get("N_BG_BENIGN") or len(ib_atq))
ib_ben = rng_bg.permutation(np.flatnonzero(g_resto == 0))[:N_BG_BENIGN]; FUNDO = X_rf[resto[np.concatenate([ib_ben, ib_atq])]]
AMK = construir_amk()
for d in DIAS: print(f"\n== Amostra do {ROT_DIA[d]}: linhas por {GRUPO_ROTULO} (estratificada; tipos pequenos entram inteiros) =="); display(tab_amostra(d, N_POR_GRUPO_RF, N_POR_GRUPO_SVM))
print(f"\nFundo (referência) do SVM: {len(FUNDO)} linhas do dia do treino, fora das explicadas: {len(ib_ben)} BENIGN + {len(ib_atq)} de ataque ({N_BG_POR_GRUPO_SVM} por tipo de ataque) = {100 * len(ib_ben) / len(FUNDO):.0f}% BENIGN, como o treino balanceado. O RF não usa fundo (o TreeExplainer usa a distribuição de treino de cada árvore).")
T_SEC["2.1_amostras"] = time.perf_counter() - t_sec0


  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 1.11 GiB); esperando até 30s e tentando de novo (1/5)...
Lendo a amostra do teste, por tipo: concluído em 3min 52s

== Amostra do dia do treino: linhas por tipo de ataque (estratificada; tipos pequenos entram inteiros) ==


,linhas disponíveis,explicadas pelo RF,explicadas pelo SVM
BENIGN,50768,300,60
DNS,4578,300,60
LDAP,4578,300,60
MSSQL,4578,300,60
NTP,4578,300,60
NetBIOS,4578,300,60
SNMP,4578,300,60
SSDP,4577,300,60
Syn,4577,300,60
TFTP,4578,300,60



== Amostra do dia do teste: linhas por tipo de ataque (estratificada; tipos pequenos entram inteiros) ==


,linhas disponíveis,explicadas pelo RF,explicadas pelo SVM
BENIGN,56216,300,60
LDAP,112491,300,60
MSSQL,854242,300,60
NetBIOS,91175,300,60
Portmap,2702,300,60
Syn,4887060,300,60
UDP,1214746,300,60
UDPLag,557,300,60



Fundo (referência) do SVM: 24 linhas do dia do treino, fora das explicadas: 12 BENIGN + 12 de ataque (1 por tipo de ataque) = 50% BENIGN, como o treino balanceado. O RF não usa fundo (o TreeExplainer usa a distribuição de treino de cada árvore).


## 3. Plano: pasta, amostras em disco e estimativa de tempo
Calcula a impressão digital da configuração (para **retomar** uma execução interrompida), guarda as amostras em disco, **mede o tempo por linha** de cada modelo e **mostra a estimativa total**. Com `SO_ESTIMAR = True` ele para aqui.

In [8]:
# ---------- 3.1 Impressão digital, pasta, amostras em disco e cronograma ----------
t_sec0 = time.perf_counter()
IMPRESSAO = hashlib.sha1(json.dumps(_limpar({"modo": MODO, "modelos": {k: (M["rotulo"], M["arquivo"].stat().st_size) for k, M in MODELOS.items()}, "n_rf": N_POR_GRUPO_RF, "n_svm": N_POR_GRUPO_SVM, "bg": N_BG_POR_GRUPO_SVM, "perm": PERMUTACOES_SVM,
                                           "semente": SEMENTE, "chunk": (CHUNK_RF, CHUNK_SVM), "ids": {d: zlib.crc32(AMOSTRA[d]["id"].astype("uint64").tobytes()) for d in DIAS}, "versoes": VERSOES}), sort_keys=True).encode()).hexdigest()[:12]
PASTA_SHAP, finalizadas = None, []
for p in sorted(PASTA_RAIZ.glob("shap_*"), reverse=True):
    try: c_ = json.loads((p / "config_shap.json").read_text(encoding="utf-8"))
    except (OSError, ValueError): continue
    if (p / "FINALIZADO").exists(): finalizadas.append((p, c_.get("impressao"))); continue
    if not NOVA_EXECUCAO and PASTA_SHAP is None and c_.get("impressao") == IMPRESSAO: PASTA_SHAP = p
if PASTA_SHAP is None and not NOVA_EXECUCAO and any(imp == IMPRESSAO for _, imp in finalizadas):
    raise RuntimeError("Já existe uma execução FINALIZADA com esta mesma configuração e os mesmos dados: " + ", ".join(p.name for p, imp in finalizadas if imp == IMPRESSAO) + ". Os resultados estão nos arquivos dessa pasta. Para refazer DE PROPÓSITO, defina NOVA_EXECUCAO = True antes desta célula.")
RETOMANDO = PASTA_SHAP is not None
if PASTA_SHAP is None:
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S"); PASTA_SHAP = PASTA_RAIZ / f"shap_{RUN_ID}"; PASTA_SHAP.mkdir(parents=True)
    salvar_json({"run_id": RUN_ID, "criado_em": datetime.now().isoformat(timespec="seconds"), "impressao": IMPRESSAO, "modo": MODO}, PASTA_SHAP / "config_shap.json")
else: RUN_ID = PASTA_SHAP.name.replace("shap_", "")
for sub in ("amostras", "chunks", "resultados"): (PASTA_SHAP / sub).mkdir(exist_ok=True)
# as amostras e o fundo (referência do SVM) ficam em disco: cada processo as lê
for d in DIAS:
    arq = PASTA_SHAP / "amostras" / f"amostra_{d}.npz"
    if not arq.exists(): np.savez_compressed(arq, X=AMOSTRA[d]["X"], y=AMOSTRA[d]["y"], grupo=AMOSTRA[d]["grupo"], id=AMOSTRA[d]["id"], rank=AMOSTRA[d]["rank"], grupos=np.array(GRUPOS_DIA[d]))
    for fam in ("RF", "SVM"):
        kk = next((k for k, M in MODELOS.items() if M["familia"] == fam), None)
        if kk is not None and not (PASTA_SHAP / "amostras" / f"X_{fam}_{d}.npy").exists(): np.save(PASTA_SHAP / "amostras" / f"X_{fam}_{d}.npy", AMK[kk][d]["X"])
if not (PASTA_SHAP / "amostras" / "fundo_svm.npy").exists(): np.save(PASTA_SHAP / "amostras" / "fundo_svm.npy", FUNDO)
print(f"Pasta da execução: {PASTA_SHAP}" + ("  (RETOMANDO)" if RETOMANDO else ""))
# cronograma: tarefas = pedaços de linhas, por modelo e dia
def pedacos(k, d): n = len(AMK[k][d]["y"]); c = CHUNK_RF if MODELOS[k]["familia"] == "RF" else CHUNK_SVM; return [(i, min(i + c, n)) for i in range(0, n, c)]
def arq_chunk(k, d, i): return PASTA_SHAP / "chunks" / k / d / f"c_{i:05d}.npz"
for k in MODELOS:
    for d in DIAS: (PASTA_SHAP / "chunks" / k / d).mkdir(parents=True, exist_ok=True)
# ---- o SVM é chamado pelo PIPELINE COMPLETO (padronização + SVM): confere isso, em vez de supor ----
def checar_padronizacao():
    from sklearn.pipeline import Pipeline
    for k, M in MODELOS.items():
        if M["familia"] != "SVM": continue
        m = M["modelo"]; assert isinstance(m, Pipeline) and list(m.named_steps) == ["escala", "svm"], f"{M['rotulo']}: o modelo não é o pipeline (escala + svm) esperado: o SHAP chamaria o SVM sem padronizar"
        esc = m.named_steps["escala"]; tr = {n_: (t_, len(c_)) for n_, t_, c_ in esc.transformers_}; num = esc.named_transformers_["num"]
        assert hasattr(num, "mean_") and hasattr(num, "scale_") and len(num.mean_) == tr["num"][1] and (num.scale_ > 0).all(), f"{M['rotulo']}: o padronizador não está ajustado"
        assert tr["num"][1] + tr["ind"][1] == D, f"{M['rotulo']}: o padronizador não cobre todos os {D} atributos"
        X = AMK[k]["teste"]["X"][:200]; via_pipe = m.decision_function(X); manual = m.named_steps["svm"].decision_function(esc.transform(X)); sem = m.named_steps["svm"].decision_function(X)
        assert np.allclose(np.asarray(via_pipe), np.asarray(manual), atol=1e-9), f"{M['rotulo']}: o pipeline não equivale a padronizar e depois prever"
        assert not np.allclose(np.asarray(via_pipe), np.asarray(sem), atol=1e-3), f"{M['rotulo']}: prever SEM padronizar dá o mesmo resultado (o padronizador não tem efeito?)"
        print(f"✔ {M['rotulo']}: o explicador chama o pipeline completo — a padronização ({tr['num'][1]} atributos numéricos, média e desvio do TREINO do próprio modelo) é aplicada ANTES do SVM em toda previsão; os {tr['ind'][1]} indicadores _ausente passam sem alteração. (Prever sem padronizar daria outro resultado: confirmado.)")
checar_padronizacao()
# ---- benchmark: mede o tempo por linha de cada modelo (3 linhas) para estimar o total ----
def _explicar_arrays(familia, m, X, fundo, perm, modo, semente=None):
    # Devolve phi (n × D × saídas), fx (n × saídas) e a linha de base (saídas). RF: TreeExplainer (exato); SVM: PermutationExplainer sobre a função de decisão, com o fundo estratificado.
    if familia == "RF":
        ex = shap.TreeExplainer(m); sv = np.asarray(ex.shap_values(X, check_additivity=False)); fx = m.predict_proba(X); base = np.atleast_1d(np.asarray(ex.expected_value, float))
        if modo == "bin": sv, fx, base = sv[:, :, 1:2], fx[:, 1:2], base[1:2]
    else:
        f = lambda Z: np.asarray(m.decision_function(Z)).reshape(len(Z), -1)
        e = shap.PermutationExplainer(f, fundo, seed=semente)(X, max_evals=2 * X.shape[1] * perm + 1); sv = np.asarray(e.values); sv = sv[:, :, None] if sv.ndim == 2 else sv        # semente fixa por pedaço: o resultado não depende da ordem nem do paralelismo
        fx = f(X); b = np.asarray(e.base_values, float); b = b.reshape(len(X), -1); base = b[0]
        assert np.allclose(b, b[0], atol=1e-9), "a linha de base do SVM varia entre as linhas"
    return sv, fx, base
BENCH = {}
print("\nMedindo o tempo por linha de cada modelo (3 linhas)...")
for k, M in MODELOS.items():
    m = M["modelo"]; X3 = AMK[k]["teste"]["X"][:3]; r0 = rss_mib(); t0 = time.perf_counter()
    if M["familia"] == "RF": shap.TreeExplainer(m); t_build = time.perf_counter() - t0
    else: t_build = 0.0
    r1 = rss_mib(); t0 = time.perf_counter(); _explicar_arrays(M["familia"], m, X3, FUNDO, PERMUTACOES_SVM, MODO); t_lin = (time.perf_counter() - t0 - (t_build if M["familia"] == "RF" else 0)) / 3
    BENCH[k] = {"t_construir_s": t_build, "t_linha_s": max(t_lin, 1e-4), "ram_extra_mib": max(rss_mib() - r0, 0.0)}
    print(f"  {M['rotulo']:13s} construir explicador {t_build:6.1f} s | {BENCH[k]['t_linha_s']:7.3f} s por linha | +{BENCH[k]['ram_extra_mib']:,.0f} MiB")
PLANO, linhas = {}, []
for k, M in MODELOS.items():
    for d in DIAS:
        for j, (a, b) in enumerate(pedacos(k, d)): PLANO[(k, d, j)] = BENCH[k]["t_construir_s"] + (b - a) * BENCH[k]["t_linha_s"]
k_est = processos_seguros(PROCESSOS_MAX, max(RAM_POR_PROCESSO_GIB, 1.2 * max(v["ram_extra_mib"] for v in BENCH.values()) / 1024 + 0.3))
for k, M in MODELOS.items():
    tot = sum(v for (kk, d, j), v in PLANO.items() if kk == k); linhas.append({"modelo": M["rotulo"], "linhas (2 dias)": sum(len(AMK[k][d]["y"]) for d in DIAS), "tarefas": sum(len(pedacos(k, d)) for d in DIAS), "s por linha": BENCH[k]["t_linha_s"], "tempo total, 1 processo": fmt_tempo(tot), f"com {k_est} processo(s)": fmt_tempo(tot / k_est)})
print(f"\n== Estimativa (1 processo → com os {k_est} que a memória livre comporta agora: {ram_livre_gib():.1f} GiB livres) =="); display(pd.DataFrame(linhas).set_index("modelo"))
print(f"Tempo total estimado: {fmt_tempo(sum(PLANO.values()))} com 1 processo, ~{fmt_tempo(sum(PLANO.values()) / k_est)} com {k_est}")
if SO_ESTIMAR: raise RuntimeError("SO_ESTIMAR = True: parei depois da estimativa. Defina SO_ESTIMAR = False para calcular.")
T_SEC["3.1_plano"] = time.perf_counter() - t_sec0


Pasta da execução: C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\shap_binario\shap_20261109_173230
✔ SVM ajustado: o explicador chama o pipeline completo — a padronização (42 atributos numéricos, média e desvio do TREINO do próprio modelo) é aplicada ANTES do SVM em toda previsão; os 1 indicadores _ausente passam sem alteração. (Prever sem padronizar daria outro resultado: confirmado.)
✔ SVM padrão: o explicador chama o pipeline completo — a padronização (42 atributos numéricos, média e desvio do TREINO do próprio modelo) é aplicada ANTES do SVM em toda previsão; os 1 indicadores _ausente passam sem alteração. (Prever sem padronizar daria outro resultado: confirmado.)

Medindo o tempo por linha de cada modelo (3 linhas)...
  RF ajustado   construir explicador    0.0 s |   0.080 s por linha | +9 MiB
  RF padrão     construir explicador    0.0 s |   0.032 s por linha | +4 MiB
  SVM ajustado  construir explicador    0.0 s |   0.632 s por linha | +0 MiB
  SVM padrão    c

,linhas (2 dias),tarefas,s por linha,"tempo total, 1 processo",com 1 processo(s)
modelo,,,,,
RF ajustado,6300,7,0.080277,8min 26s,8min 26s
RF padrão,6300,7,0.031796,3min 20s,3min 20s
SVM ajustado,1260,32,0.631751,13min 16s,13min 16s
SVM padrão,1260,32,0.665193,13min 58s,13min 58s


Tempo total estimado: 39min 01s com 1 processo, ~39min 01s com 1


## 4. Valores SHAP (a parte longa)
Explica as linhas em **tarefas independentes, cada uma num processo à parte**, com **vários ao mesmo tempo só se houver memória livre**. Mostra uma **barra de progresso** ponderada pelo tempo estimado. Cada tarefa é salva ao terminar: **se interromper, rode a célula de novo**. No fim confere a **aditividade** em todas as linhas.

In [9]:
# ---------- 4.1 Cálculo dos valores SHAP (paralelo, só se houver memória; retomável) ----------
t_sec0 = time.perf_counter()
def _worker_chunk(familia, caminho_modelo, caminho_X, a, b, caminho_fundo, perm, modo, saida):
    # Um processo NOVO por tarefa: carrega o modelo, explica as linhas [a, b) e grava o resultado antes de devolver os tempos.
    import shap
    m = joblib.load(caminho_modelo); X = np.load(caminho_X, mmap_mode="r")[a:b].copy(); fundo = np.load(caminho_fundo) if (familia == "SVM" and caminho_fundo) else None
    with Medidor() as med: sv, fx, base = _explicar_arrays(familia, m, X, fundo, perm, modo, SEMENTE + int(a))
    tmp = Path(str(saida) + ".tmp.npz"); np.savez_compressed(tmp, phi=sv.astype(np.float32), fx=fx.astype(np.float32), base=np.asarray(base, float), a=np.int64(a), b=np.int64(b)); os.replace(tmp, saida)
    return {"n": int(b - a), "tempo_s": med.relogio, "cpu_s": med.cpu, "pico_ram_mib": med.pico_mib}, None
TAREFAS = [(k, d, j) for k in MODELOS for d in DIAS for j in range(len(pedacos(k, d)))]
pend = sorted([t for t in TAREFAS if not arq_chunk(t[0], t[1], pedacos(t[0], t[1])[t[2]][0]).exists()], key=lambda t: -PLANO[t])               # as maiores primeiro (equilibra os processos)
print(f"Tarefas prontas: {len(TAREFAS) - len(pend)}/{len(TAREFAS)}" + ("  (retomando)" if len(pend) < len(TAREFAS) else "") + f" | a calcular agora: {len(pend)}")
STATS_RUN = {}
if pend:
    peso_total = int(sum(PLANO[t] for t in pend)) + 1; barra = Barra(peso_total, "Valores SHAP"); soma_tarefas, t_sessao, feitas = 0.0, time.perf_counter(), []
    print(f"Memória livre agora: {ram_livre_gib():.1f} GiB → começo com {processos_seguros(PROCESSOS_MAX, RAM_POR_PROCESSO_GIB)} processo(s) (teto {PROCESSOS_MAX}; reavalio a memória a cada partida). A barra é ponderada pelo tempo estimado de cada tarefa.")
    def lancar(t):
        k, d, j = t; M = MODELOS[k]; a, b = pedacos(k, d)[j]; ex = LokyPool(max_workers=1)
        return ex, ex.submit(_worker_chunk, M["familia"], str(M["arquivo"]), str(PASTA_SHAP / "amostras" / f"X_{M['familia']}_{d}.npy"), a, b, str(PASTA_SHAP / "amostras" / "fundo_svm.npy"), PERMUTACOES_SVM, MODO, str(arq_chunk(k, d, a)))
    def coletar(t, res, k_ini):
        global soma_tarefas
        assert res is not None, f"a tarefa {t} passou do limite de {fmt_tempo(LIMITE_TAREFA_S)}: aumente LIMITE_TAREFA_S ou diminua CHUNK_SVM"
        meta, _ = res; soma_tarefas += meta["tempo_s"]; feitas.append({**meta, "modelo": t[0], "dia": t[1], "tarefa": t[2], "simultaneos_na_partida": k_ini})
        barra.avancar(int(PLANO[t]), f"{MODELOS[t[0]]['rotulo']} | {ROT_DIA[t[1]]} | tarefa {len(feitas)}/{len(pend)} ({meta['n']} linhas em {fmt_tempo(meta['tempo_s'])}, {k_ini} simultâneo(s) na partida)")
    try: STATS_RUN = rodar_tarefas(pend, lancar, coletar, barra, LIMITE_TAREFA_S, PROCESSOS_MAX, max(RAM_POR_PROCESSO_GIB, 1.2 * max(v["ram_extra_mib"] for v in BENCH.values()) / 1024 + 0.3), "SHAP", PARAR_APOS_TAREFAS)
    finally:
        rel = time.perf_counter() - t_sessao
        if feitas: pd.DataFrame(feitas).assign(sessao_inicio=datetime.now().isoformat(timespec="seconds")).to_csv(PASTA_SHAP / "resultados" / f"tempos_tarefas_{datetime.now():%Y%m%d_%H%M%S}.csv", index=False)
    barra.fechar(); STATS_RUN = {**STATS_RUN, "relogio_s": rel, "soma_tempos_tarefas_s": soma_tarefas, "ganho_paralelismo": soma_tarefas / rel if rel > 0 else float("nan")}
    print(f"✔ {len(feitas)} tarefas em {fmt_tempo(rel)} | no máximo {STATS_RUN['max_simultaneos']} processo(s) simultâneo(s) | ganho observado do paralelismo: {STATS_RUN['ganho_paralelismo']:.2f}× | memória por processo medida: ~{STATS_RUN['ram_proc_final']:.2f} GiB" + (f" | tarefas interrompidas por memória: {STATS_RUN['interrompidos_por_memoria']}" if STATS_RUN["interrompidos_por_memoria"] else ""))
# ---- junta os pedaços na ordem original ----
PHI, FX, BASE, PRED = {k: {} for k in MODELOS}, {k: {} for k in MODELOS}, {k: {} for k in MODELOS}, {k: {} for k in MODELOS}
for k, M in MODELOS.items():
    for d in DIAS:
        zs = [np.load(arq_chunk(k, d, a)) for a, b in pedacos(k, d)]; PHI[k][d] = np.concatenate([z["phi"] for z in zs]).astype(np.float64); FX[k][d] = np.concatenate([z["fx"] for z in zs]).astype(np.float64); BASE[k][d] = zs[0]["base"]
        assert all(np.allclose(z["base"], BASE[k][d]) for z in zs), f"a linha de base varia entre os pedaços ({k}, {d})"
        assert len(PHI[k][d]) == len(AMK[k][d]["y"]) and np.isfinite(PHI[k][d]).all() and np.isfinite(FX[k][d]).all(), f"valores SHAP incompletos ou não finitos ({k}, {d})"
        PRED[k][d] = M["rotular"](FX[k][d])
ADIT = {(k, d): float(np.abs(BASE[k][d][None, :] + PHI[k][d].sum(axis=1) - FX[k][d]).max()) for k in MODELOS for d in DIAS}
print("\nAditividade (linha de base + soma dos valores SHAP = saída do modelo; erro máximo): " + " | ".join(f"{MODELOS[k]['rotulo']} {ROT_DIA[d][4:]}: {v:.1e}" for (k, d), v in ADIT.items()))
T_SEC["4.1_shap"] = time.perf_counter() - t_sec0


Tarefas prontas: 0/78 | a calcular agora: 78


Memória livre agora: 0.6 GiB → começo com 1 processo(s) (teto 4; reavalio a memória a cada partida). A barra é ponderada pelo tempo estimado de cada tarefa.
Valores SHAP: concluído em 36min 14s
✔ 78 tarefas em 36min 13s | no máximo 1 processo(s) simultâneo(s) | ganho observado do paralelismo: 0.88× | memória por processo medida: ~1.00 GiB

Aditividade (linha de base + soma dos valores SHAP = saída do modelo; erro máximo): RF ajustado do treino: 3.7e-08 | RF ajustado do teste: 3.4e-08 | RF padrão do treino: 3.6e-08 | RF padrão do teste: 3.9e-08 | SVM ajustado do treino: 6.3e-07 | SVM ajustado do teste: 2.6e-07 | SVM padrão do treino: 1.5e-07 | SVM padrão do teste: 1.2e-07


## 5. Importância global e direção
A participação de cada atributo no total de |SHAP| (peso igual para cada tipo de ataque), por classe, e a **direção** (correlação entre o valor do atributo e o SHAP).

In [10]:
import warnings
# ---------- 5.1 Importância global e direção ----------
t_sec0 = time.perf_counter()
def phi_proprio(k, d):
    # Valores SHAP da saída que se quer explicar em cada linha: multiclasse = a classe REAL da linha; binário = a saída "ataque".
    P, A = PHI[k][d], AMK[k][d]
    return P[np.arange(len(P)), :, A["y"]] if MODO == "multi" else P[:, :, 0]
def partilha(imp): return imp / np.nansum(imp, axis=1, keepdims=True)
def imp_grupos(phi, grupo, G):
    imp = np.full((G, D), np.nan)
    for g in range(G):
        m = grupo == g
        if m.any(): imp[g] = np.abs(phi[m]).mean(axis=0)
    return imp
def partilha_geral(phi, lista_idx):
    # participação de cada atributo no total de |SHAP|, com PESO IGUAL para cada grupo (os grupos minoritários contam o mesmo que os grandes)
    s = [np.abs(phi[ix]).mean(axis=0) for ix in lista_idx if len(ix)]; s = np.array([v / v.sum() for v in s if v.sum() > 0]); return s.mean(axis=0)
def corr_postos(x, p):
    if len(x) < 5: return np.full(x.shape[1], np.nan)
    rx, rp = rankdata(x, axis=0), rankdata(p, axis=0); rx, rp = rx - rx.mean(0), rp - rp.mean(0); den = np.sqrt((rx ** 2).sum(0) * (rp ** 2).sum(0)); return np.divide((rx * rp).sum(0), den, out=np.full(x.shape[1], np.nan), where=den > 0)
PHIP, IMP, SHARE, GERAL, DIRE, GIDX = {k: {} for k in MODELOS}, {k: {} for k in MODELOS}, {k: {} for k in MODELOS}, {k: {} for k in MODELOS}, {k: {} for k in MODELOS}, {k: {} for k in MODELOS}
for k in MODELOS:
    for d in DIAS: GIDX[k][d] = [np.flatnonzero(AMK[k][d]["grupo"] == i) for i in range(len(GRUPOS_DIA[d]))]
for k in MODELOS:
    for d in DIAS:
        A = AMK[k][d]; G = len(GRUPOS_DIA[d]); phi = phi_proprio(k, d); PHIP[k][d] = phi; IMP[k][d] = imp_grupos(phi, A["grupo"], G); SHARE[k][d] = partilha(IMP[k][d]); GERAL[k][d] = partilha_geral(phi, GIDX[k][d])
        DIRE[k][d] = np.array([corr_postos(A["X"][ix], phi[ix]) if len(ix) else np.full(D, np.nan) for ix in GIDX[k][d]])
def top_geral(k, d, n=15):
    s = GERAL[k][d]; o = np.argsort(-s)[:n]
    with warnings.catch_warnings(): warnings.simplefilter("ignore", RuntimeWarning); dir_ = np.nanmean(DIRE[k][d], axis=0); imp_ = np.nanmean(IMP[k][d], axis=0)      # atributos constantes (indicadores sempre 0) não têm direção
    return pd.DataFrame({"atributo": [CM[j] for j in o], "participação (%)": 100 * s[o], "direção (corr. valor × SHAP, média dos grupos)": dir_[o], "mean |SHAP| (média dos grupos)": imp_[o]}).set_index("atributo")
for k, M in MODELOS.items():
    for d in DIAS: print(f"\n== {M['rotulo']} | {ROT_DIA[d]}: os {TOP_K + 5} atributos que mais pesam (participação no total de |SHAP|; peso igual para cada {GRUPO_ROTULO}) =="); display(top_geral(k, d, TOP_K + 5).round(4))
print(f"\n(direção > 0: valores altos do atributo aumentam a saída explicada; < 0: diminuem. Unidades: RF = probabilidade; SVM = função de decisão. Por isso a comparação entre RF e SVM usa a PARTICIPAÇÃO (%), não o valor absoluto.)")
TOPG = {}
for k, M in MODELOS.items():
    for d in DIAS:
        linhas = []
        for gi, g in enumerate(GRUPOS_DIA[d]):
            if np.isnan(SHARE[k][d][gi]).all(): continue
            o = np.argsort(-np.nan_to_num(SHARE[k][d][gi]))[:5]; linhas.append({GRUPO_ROTULO: g, "linhas": len(GIDX[k][d][gi]), **{f"#{r + 1}": f"{CM[j]} ({100 * SHARE[k][d][gi][j]:.0f}%)" for r, j in enumerate(o)}})
        TOPG[(k, d)] = pd.DataFrame(linhas).set_index(GRUPO_ROTULO); print(f"\n-- {M['rotulo']} | {ROT_DIA[d]}: os 5 atributos que mais pesam em cada {GRUPO_ROTULO} --"); display(TOPG[(k, d)])
T_SEC["5.1_importancia"] = time.perf_counter() - t_sec0



== RF ajustado | dia do treino: os 15 atributos que mais pesam (participação no total de |SHAP|; peso igual para cada tipo de ataque) ==


,participação (%),"direção (corr. valor × SHAP, média dos grupos)",mean |SHAP| (média dos grupos)
atributo,,,
avg_fwd_segment_size,8.3685,0.1312,0.0472
bwd_header_length,7.5941,-0.3072,0.0433
bwd_iat_max,5.8186,-0.3508,0.0330
protocol,5.6088,0.3196,0.0329
bwd_iat_mean,5.5540,-0.3357,0.0315
bwd_iat_total,4.9571,-0.3352,0.0281
total_length_of_fwd_packets,4.6781,0.3995,0.0265
fwd_packet_length_max,4.4080,0.2607,0.0250
urg_flag_count,4.2587,-0.2950,0.0251



== RF ajustado | dia do teste: os 15 atributos que mais pesam (participação no total de |SHAP|; peso igual para cada tipo de ataque) ==


,participação (%),"direção (corr. valor × SHAP, média dos grupos)",mean |SHAP| (média dos grupos)
atributo,,,
avg_fwd_segment_size,9.2479,0.3076,0.0537
bwd_header_length,7.1766,-0.6088,0.0411
protocol,6.7655,0.5063,0.0404
bwd_iat_max,5.5745,-0.5591,0.0320
bwd_iat_mean,5.4755,-0.5471,0.0315
urg_flag_count,5.2523,-0.5714,0.0315
bwd_iat_total,4.7585,-0.5483,0.0273
fwd_packet_length_max,4.6968,0.3549,0.0273
init_win_bytes_forward,4.5710,-0.5568,0.0270



== RF padrão | dia do treino: os 15 atributos que mais pesam (participação no total de |SHAP|; peso igual para cada tipo de ataque) ==


,participação (%),"direção (corr. valor × SHAP, média dos grupos)",mean |SHAP| (média dos grupos)
atributo,,,
avg_fwd_segment_size,8.7678,0.1541,0.0492
bwd_header_length,8.2895,-0.2937,0.0468
fwd_packet_length_max,6.0929,0.2601,0.0343
bwd_iat_max,5.2858,-0.3612,0.0298
bwd_iat_mean,5.2556,-0.3234,0.0294
total_length_of_fwd_packets,5.2205,0.2194,0.0294
protocol,5.0241,0.3196,0.0292
urg_flag_count,4.6871,-0.2950,0.0274
init_win_bytes_forward,4.1870,-0.1592,0.0245


... (22 outputs left out)


-- SVM padrão | dia do treino: os 5 atributos que mais pesam em cada tipo de ataque --


,linhas,#1,#2,#3,#4,#5
tipo de ataque,,,,,,
BENIGN,60,bwd_packet_length_min (27%),protocol (14%),down_up_ratio (7%),urg_flag_count (6%),ack_flag_count (5%)
DNS,60,protocol (23%),bwd_packet_length_min (15%),down_up_ratio (8%),urg_flag_count (8%),avg_fwd_segment_size (7%)
LDAP,60,protocol (22%),bwd_packet_length_min (14%),avg_fwd_segment_size (13%),down_up_ratio (8%),urg_flag_count (7%)
MSSQL,60,protocol (25%),bwd_packet_length_min (17%),down_up_ratio (9%),urg_flag_count (8%),avg_bwd_segment_size (7%)
NTP,60,protocol (24%),bwd_packet_length_min (17%),down_up_ratio (10%),urg_flag_count (8%),avg_bwd_segment_size (7%)
NetBIOS,60,protocol (24%),bwd_packet_length_min (16%),down_up_ratio (9%),urg_flag_count (8%),avg_bwd_segment_size (6%)
SNMP,60,protocol (22%),avg_fwd_segment_size (14%),bwd_packet_length_min (13%),down_up_ratio (8%),urg_flag_count (7%)
SSDP,60,protocol (25%),bwd_packet_length_min (18%),down_up_ratio (11%),urg_flag_count (8%),avg_bwd_segment_size (7%)
Syn,60,ack_flag_count (25%),protocol (17%),bwd_packet_length_min (8%),urg_flag_count (7%),idle_std (4%)



-- SVM padrão | dia do teste: os 5 atributos que mais pesam em cada tipo de ataque --


,linhas,#1,#2,#3,#4,#5
tipo de ataque,,,,,,
BENIGN,60,protocol (18%),bwd_packet_length_min (18%),urg_flag_count (9%),down_up_ratio (7%),avg_bwd_segment_size (5%)
LDAP,60,protocol (15%),min_seg_size_forward (13%),avg_fwd_segment_size (12%),bwd_packet_length_min (11%),down_up_ratio (7%)
MSSQL,60,protocol (19%),bwd_packet_length_min (15%),min_seg_size_forward (11%),down_up_ratio (10%),urg_flag_count (7%)
NetBIOS,60,protocol (18%),bwd_packet_length_min (16%),min_seg_size_forward (13%),down_up_ratio (12%),urg_flag_count (6%)
Portmap,60,protocol (19%),bwd_packet_length_min (14%),min_seg_size_forward (10%),down_up_ratio (9%),urg_flag_count (7%)
Syn,60,ack_flag_count (23%),protocol (20%),bwd_packet_length_min (9%),urg_flag_count (8%),fwd_packets_s (5%)
UDP,60,protocol (25%),bwd_packet_length_min (18%),down_up_ratio (11%),urg_flag_count (9%),avg_bwd_segment_size (7%)
UDPLag,60,protocol (24%),bwd_packet_length_min (15%),urg_flag_count (9%),down_up_ratio (9%),ack_flag_count (9%)


## 6. A amostra basta? Estabilidade, convergência e intervalos
Mede se a importância global é estável (metades independentes, frações da amostra e bootstrap), pelas regras fixadas na introdução.

In [11]:
# ---------- 6.1 A amostra basta? Estabilidade, convergência e intervalos (bootstrap) ----------
t_sec0 = time.perf_counter()
def spearman(a, b):
    ra, rb = rankdata(a) - rankdata(a).mean(), rankdata(b) - rankdata(b).mean(); den = math.sqrt((ra ** 2).sum() * (rb ** 2).sum()); return float((ra * rb).sum() / den) if den > 0 else float("nan")
def top_sobrep(a, b, n=TOP_K): return len(set(np.argsort(-a)[:n]) & set(np.argsort(-b)[:n]))
rng_e = np.random.default_rng(SEMENTE + 99); ESTAB, ESTAB_G, CONV, BOOT = [], [], [], {}
FRACS = (0.1, 0.25, 0.5, 0.75)
barra = Barra(len(MODELOS) * len(DIAS), "Estabilidade da amostra")
for k, M in MODELOS.items():
    for d in DIAS:
        phi, idx = PHIP[k][d], GIDX[k][d]; G = len(idx); rho_g, top_g, rho_gr = [], [], [[] for _ in range(G)]
        for r in range(N_SPLITS_ESTAB):                                                      # metades independentes (dentro de cada grupo), com o mesmo tamanho
            A_, B_ = [], []
            for gi, ix in enumerate(idx):
                p = rng_e.permutation(ix); h = len(p) // 2; A_.append(p[:h]); B_.append(p[h:2 * h])
                if h >= 3:
                    sa, sb = np.abs(phi[A_[-1]]).mean(0), np.abs(phi[B_[-1]]).mean(0); rho_gr[gi].append(spearman(sa, sb))
            sA, sB = partilha_geral(phi, A_), partilha_geral(phi, B_); rho_g.append(spearman(sA, sB)); top_g.append(top_sobrep(sA, sB))
        full = GERAL[k][d]; conv = {}
        for fr in FRACS:
            v = []
            for r in range(20):
                sub = [rng_e.choice(ix, size=max(2, int(math.ceil(fr * len(ix)))), replace=False) if len(ix) > 2 else ix for ix in idx]; v.append(spearman(partilha_geral(phi, sub), full))
            conv[fr] = float(np.nanmean(v)); CONV.append({"modelo": M["rotulo"], "dia": ROT_DIA[d], "fração da amostra": fr, "linhas por dia": int(fr * len(phi)), "correlação de postos com a amostra inteira": conv[fr]})
        CONV.append({"modelo": M["rotulo"], "dia": ROT_DIA[d], "fração da amostra": 1.0, "linhas por dia": len(phi), "correlação de postos com a amostra inteira": 1.0})
        bs = np.array([partilha_geral(phi, [rng_e.choice(ix, size=len(ix), replace=True) for ix in idx if len(ix)]) for _ in range(N_BOOT)]); lo, hi = np.percentile(bs, [2.5, 97.5], axis=0); BOOT[(k, d)] = (lo, hi)
        o = np.argsort(-full)[:TOP_K]; larg = float(np.mean((hi[o] - lo[o]) / (2 * np.maximum(full[o], 1e-12))))
        ESTAB.append({"modelo": M["rotulo"], "dia": ROT_DIA[d], "linhas": len(phi), "split-half: correlação (média)": float(np.nanmean(rho_g)), "split-half: correlação (mínima)": float(np.nanmin(rho_g)), f"split-half: sobreposição do top {TOP_K} (média)": float(np.mean(top_g)),
                      "convergência com 50% da amostra": conv[0.5], f"IC95% bootstrap: meia-largura relativa média do top {TOP_K}": larg})
        for gi, g in enumerate(GRUPOS_DIA[d]): ESTAB_G.append({"modelo": M["rotulo"], "dia": ROT_DIA[d], GRUPO_ROTULO: g, "linhas": len(idx[gi]), "split-half: correlação (média)": float(np.nanmean(rho_gr[gi])) if rho_gr[gi] else np.nan})
        barra.avancar(1, f"{M['rotulo']} | {ROT_DIA[d]}")
barra.fechar()
TAB_ESTAB, TAB_ESTAB_G, TAB_CONV = pd.DataFrame(ESTAB), pd.DataFrame(ESTAB_G), pd.DataFrame(CONV)
c1, c2, c3 = "split-half: correlação (média)", f"split-half: sobreposição do top {TOP_K} (média)", "convergência com 50% da amostra"
TAB_ESTAB["amostra suficiente?"] = ["SIM" if (a >= REGRAS_ESTAB["rho_metade_geral"] and b >= REGRAS_ESTAB["sobreposicao_top10_geral"] and c >= REGRAS_ESTAB["rho_convergencia_50pct"]) else "ATENÇÃO: aumentar a amostra" for a, b, c in zip(TAB_ESTAB[c1], TAB_ESTAB[c2], TAB_ESTAB[c3])]
TAB_ESTAB_G["situação"] = ["estável" if (np.isfinite(r) and r >= REGRAS_ESTAB["rho_metade_grupo"]) else ("poucas linhas" if not np.isfinite(r) else "instável") for r in TAB_ESTAB_G[c1]]
print("== A amostra basta? Estabilidade da importância global (cada metade da amostra, sorteada de novo " + f"{N_SPLITS_ESTAB} vezes, deve dar o mesmo ranking; as metades são MENORES que a amostra, então o teste é conservador) ==")
display(TAB_ESTAB.set_index(["modelo", "dia"]).round(3))
print(f"\nRegras (fixadas antes): correlação de postos entre metades ≥ {REGRAS_ESTAB['rho_metade_geral']}, sobreposição do top {TOP_K} ≥ {REGRAS_ESTAB['sobreposicao_top10_geral']}, convergência com 50% da amostra ≥ {REGRAS_ESTAB['rho_convergencia_50pct']}; por {GRUPO_ROTULO}: correlação entre metades ≥ {REGRAS_ESTAB['rho_metade_grupo']}.")
print("\n== Convergência: correlação de postos entre a importância com uma fração da amostra e com a amostra inteira =="); display(TAB_CONV.pivot_table(index=["modelo", "dia"], columns="fração da amostra", values="correlação de postos com a amostra inteira").round(3))
inst = TAB_ESTAB_G[TAB_ESTAB_G["situação"] != "estável"]
print(f"\nGrupos ({GRUPO_ROTULO}) com importância pouco estável entre metades: " + (", ".join(f"{r['modelo']}/{r['dia'][4:]}/{r[GRUPO_ROTULO]} (ρ={r[c1]:.2f}, {r['linhas']} linhas)" for _, r in inst.iterrows()) if len(inst) else "nenhum"))
T_SEC["6.1_estabilidade"] = time.perf_counter() - t_sec0


Estabilidade da amostra: concluído em 5s
== A amostra basta? Estabilidade da importância global (cada metade da amostra, sorteada de novo 50 vezes, deve dar o mesmo ranking; as metades são MENORES que a amostra, então o teste é conservador) ==


linhas  split-half: correlação (média)  split-half: correlação (mínima)  split-half: sobreposição do top 10 (média)  convergência com 50% da amostra  \
modelo       dia                                                                                                                                                                   
RF ajustado  dia do treino    3900                           0.999                            0.998                                       10.00                            1.000   
             dia do teste     2400                           0.999                            0.999                                       10.00                            1.000   
RF padrão    dia do treino    3900                           1.000                            0.998                                       10.00                            1.000   
             dia do teste     2400                           0.999                            0.997                                       10.00                            0.999   
SVM ajustado dia do treino     780                           0.993                            0.988                                        9.12                            0.997   
             dia do teste      480                           0.992                            0.984                                        9.94                            0.997   
SVM padrão   dia do treino     780                           0.991                            0.982                                       10.00                            0.997   
             dia do teste      480                           0.991                            0.968                                       10.00                            0.996   

                            IC95% bootstrap: meia-largura relativa média do top 10 amostra suficiente?  
modelo       dia                                                                                        
RF ajustado  dia do treino                                                   0.009                 SIM  
             dia do teste                                                    0.012                 SIM  
RF padrão    dia do treino                                                   0.008                 SIM  
             dia do teste                                                    0.012                 SIM  
SVM ajustado dia do treino                                                   0.051                 SIM  
             dia do teste                                                    0.081                 SIM  
SVM padrão   dia do treino                                                   0.029                 SIM  
             dia do teste                                                    0.050                 SIM


Regras (fixadas antes): correlação de postos entre metades ≥ 0.9, sobreposição do top 10 ≥ 7, convergência com 50% da amostra ≥ 0.95; por tipo de ataque: correlação entre metades ≥ 0.8.

== Convergência: correlação de postos entre a importância com uma fração da amostra e com a amostra inteira ==


fração da amostra            0.10   0.25   0.50   0.75  1.00
modelo       dia                                            
RF ajustado  dia do teste   0.999  0.999  1.000  1.000   1.0
             dia do treino  0.999  0.999  1.000  1.000   1.0
RF padrão    dia do teste   0.997  0.999  0.999  1.000   1.0
             dia do treino  0.999  1.000  1.000  1.000   1.0
SVM ajustado dia do teste   0.986  0.994  0.997  0.999   1.0
             dia do treino  0.985  0.994  0.997  0.998   1.0
SVM padrão   dia do teste   0.982  0.992  0.996  0.999   1.0
             dia do treino  0.982  0.993  0.997  0.998   1.0


Grupos (tipo de ataque) com importância pouco estável entre metades: nenhum


## 7 a 9. Dia do treino × dia do teste, RF × SVM e acertos × erros
**7.1** compara a importância dos atributos nos dois dias e mostra os que mais mudaram, com a mediana do valor em cada dia. **8.1** compara o RF com o SVM. **9.1** mostra, no teste, o que separa as linhas que o modelo acerta das que erra.

In [12]:
# ---------- 7.1 Dia do treino × dia do teste: o que muda nas explicações ----------
t_sec0 = time.perf_counter()
GC = [g for g in GRUPOS_DIA["treino"] if g in GRUPOS_DIA["teste"]]; gi_ = {d: {g: i for i, g in enumerate(GRUPOS_DIA[d])} for d in DIAS}
DIAS_LONGO, DIAS_RESUMO = [], []
for k, M in MODELOS.items():
    sp = {"geral": spearman(GERAL[k]["treino"], GERAL[k]["teste"])}
    for g in GC:
        a, b = SHARE[k]["treino"][gi_["treino"][g]], SHARE[k]["teste"][gi_["teste"][g]]
        if np.isnan(a).any() or np.isnan(b).any(): continue
        sp[g] = spearman(a, b); delta = b - a; o = np.argsort(-np.abs(delta))[:5]; ix_t, ix_e = GIDX[k]["treino"][gi_["treino"][g]], GIDX[k]["teste"][gi_["teste"][g]]
        for r, j in enumerate(o): DIAS_LONGO.append({"modelo": M["rotulo"], GRUPO_ROTULO: g, "posição": r + 1, "atributo": CM[j], "participação treino (%)": 100 * a[j], "participação teste (%)": 100 * b[j], "Δ (p.p.)": 100 * delta[j],
                                                       "mediana do atributo (treino)": float(np.median(AMK[k]["treino"]["X"][ix_t][:, j])), "mediana do atributo (teste)": float(np.median(AMK[k]["teste"]["X"][ix_e][:, j])),
                                                       "SHAP médio (treino)": float(PHIP[k]["treino"][ix_t][:, j].mean()), "SHAP médio (teste)": float(PHIP[k]["teste"][ix_e][:, j].mean())})
    DIAS_RESUMO.append({"modelo": M["rotulo"], **{f"ρ {g}": v for g, v in sp.items()}})
TAB_DIAS_LONGO, TAB_DIAS_RESUMO = pd.DataFrame(DIAS_LONGO), pd.DataFrame(DIAS_RESUMO).set_index("modelo")
print(f"== Correlação de postos (ρ) entre a importância no dia do treino e no dia do teste (1 = mesmo ranking de atributos; só os grupos presentes nos dois dias) =="); display(TAB_DIAS_RESUMO.round(3))
for k, M in MODELOS.items():
    d_ = TAB_DIAS_LONGO[TAB_DIAS_LONGO["modelo"] == M["rotulo"]]
    if len(d_): print(f"\n-- {M['rotulo']}: o atributo que mais mudou de participação em cada {GRUPO_ROTULO} (treino → teste) --"); display(d_[d_["posição"] == 1].drop(columns=["modelo", "posição"]).set_index(GRUPO_ROTULO).round(3))
T_SEC["7.1_dias"] = time.perf_counter() - t_sec0
# ---------- 8.1 RF × SVM: eles se apoiam nos mesmos atributos? ----------
t_sec0 = time.perf_counter(); PARES = [(a, b) for a, b in (("rf_ajustado", "svm_ajustado"), ("rf_padrao", "svm_padrao")) if a in MODELOS and b in MODELOS]; LINHAS_MOD = []
for a, b in PARES:
    for d in DIAS:
        r = {"par": f"{MODELOS[a]['rotulo']} × {MODELOS[b]['rotulo']}", "dia": ROT_DIA[d], "ρ geral": spearman(GERAL[a][d], GERAL[b][d]), f"top {TOP_K} em comum": top_sobrep(GERAL[a][d], GERAL[b][d])}
        for g in GRUPOS_DIA[d]:
            sa, sb = SHARE[a][d][gi_[d][g]], SHARE[b][d][gi_[d][g]]
            if not (np.isnan(sa).any() or np.isnan(sb).any()): r[f"ρ {g}"] = spearman(sa, sb)
        LINHAS_MOD.append(r)
TAB_MOD = pd.DataFrame(LINHAS_MOD).set_index(["par", "dia"]); print("== Concordância entre o RF e o SVM na importância dos atributos (ρ = correlação de postos; participações em %, comparáveis apesar das unidades diferentes) =="); display(TAB_MOD.round(3))
for a, b in PARES:
    d = "teste"; ta, tb = np.argsort(-GERAL[a][d])[:TOP_K], np.argsort(-GERAL[b][d])[:TOP_K]
    print(f"\n{ROT_DIA[d]} | top {TOP_K}: {MODELOS[a]['rotulo']}: " + ", ".join(CM[j] for j in ta[:6]) + f" ... | {MODELOS[b]['rotulo']}: " + ", ".join(CM[j] for j in tb[:6]) + f" ... | em comum: {sorted(CM[j] for j in set(ta) & set(tb))}")
T_SEC["8.1_rf_svm"] = time.perf_counter() - t_sec0
# ---------- 9.1 No dia do teste: o que separa as linhas que o modelo acerta das que ele erra ----------
t_sec0 = time.perf_counter(); ERROS = []
for k, M in MODELOS.items():
    d = "teste"; A = AMK[k][d]; ok_ = PRED[k][d] == A["y"]
    for gi, g in enumerate(GRUPOS_DIA[d]):
        ix = GIDX[k][d][gi]; c, e = ix[ok_[ix]], ix[~ok_[ix]]
        if len(c) < 10 or len(e) < 10: continue
        dif = PHIP[k][d][e].mean(0) - PHIP[k][d][c].mean(0); o = np.argsort(-np.abs(dif))[:5]
        for r, j in enumerate(o): ERROS.append({"modelo": M["rotulo"], GRUPO_ROTULO: g, "acertos": len(c), "erros": len(e), "posição": r + 1, "atributo": CM[j], "SHAP médio nos acertos": float(PHIP[k][d][c][:, j].mean()), "SHAP médio nos erros": float(PHIP[k][d][e][:, j].mean()), "Δ (erros − acertos)": float(dif[j]),
                                                 "mediana do atributo (acertos)": float(np.median(A["X"][c][:, j])), "mediana do atributo (erros)": float(np.median(A["X"][e][:, j]))})
TAB_ERROS = pd.DataFrame(ERROS)
print("== Dia do teste: nos grupos com ao menos 10 acertos e 10 erros na amostra, o atributo que mais separa os erros dos acertos (diferença do SHAP médio; negativo = empurra a saída explicada para baixo nos erros) ==")
if len(TAB_ERROS):
    for k, M in MODELOS.items():
        d_ = TAB_ERROS[(TAB_ERROS["modelo"] == M["rotulo"]) & (TAB_ERROS["posição"] == 1)]
        if len(d_): print(f"\n-- {M['rotulo']} --"); display(d_.drop(columns=["modelo", "posição"]).set_index(GRUPO_ROTULO).round(3))
else: print("(nenhum grupo tem 10 acertos e 10 erros na amostra)")
T_SEC["9.1_erros"] = time.perf_counter() - t_sec0


== Correlação de postos (ρ) entre a importância no dia do treino e no dia do teste (1 = mesmo ranking de atributos; só os grupos presentes nos dois dias) ==


,ρ geral,ρ BENIGN,ρ LDAP,ρ MSSQL,ρ NetBIOS,ρ Syn,ρ UDP,ρ UDPLag
modelo,,,,,,,,
RF ajustado,0.977,0.963,0.977,0.986,0.989,0.749,0.994,0.720
RF padrão,0.961,0.964,0.968,0.968,0.982,0.800,0.996,0.768
SVM ajustado,0.931,0.979,0.967,0.952,0.878,0.916,0.994,0.925
SVM padrão,0.953,0.984,0.960,0.950,0.899,0.933,0.998,0.910



-- RF ajustado: o atributo que mais mudou de participação em cada tipo de ataque (treino → teste) --


,atributo,participação treino (%),participação teste (%),Δ (p.p.),mediana do atributo (treino),mediana do atributo (teste),SHAP médio (treino),SHAP médio (teste)
tipo de ataque,,,,,,,,
BENIGN,min_seg_size_forward,1.237,3.404,2.166,20.0,2.000000e+01,-0.003,0.013
LDAP,min_seg_size_forward,2.517,0.714,-1.803,236.0,-1.062719e+09,0.012,-0.000
MSSQL,min_seg_size_forward,3.094,1.191,-1.903,459.0,-1.062719e+09,0.016,0.003
NetBIOS,min_seg_size_forward,1.552,0.532,-1.020,20.0,-1.062719e+09,0.005,-0.003
Syn,avg_fwd_segment_size,0.951,6.200,5.249,0.0,6.000000e+00,-0.002,-0.044
UDP,min_seg_size_forward,0.359,0.173,-0.186,8.0,8.000000e+00,0.000,-0.001
UDPLag,avg_fwd_segment_size,3.222,8.107,4.884,0.0,3.770000e+02,0.012,0.028



-- RF padrão: o atributo que mais mudou de participação em cada tipo de ataque (treino → teste) --


,atributo,participação treino (%),participação teste (%),Δ (p.p.),mediana do atributo (treino),mediana do atributo (teste),SHAP médio (treino),SHAP médio (teste)
tipo de ataque,,,,,,,,
BENIGN,urg_flag_count,5.547,7.651,2.103,0.000,5.000000e-01,-0.020,-0.037
LDAP,min_seg_size_forward,2.218,0.652,-1.566,236.000,-1.062719e+09,0.010,-0.000
MSSQL,min_seg_size_forward,2.827,1.090,-1.737,459.000,-1.062719e+09,0.014,0.003
NetBIOS,min_seg_size_forward,1.437,0.452,-0.985,20.000,-1.062719e+09,0.004,-0.002
Syn,flow_iat_std,8.928,2.612,-6.316,8688427.061,0.000000e+00,0.059,0.004
UDP,min_seg_size_forward,0.428,0.283,-0.145,8.000,8.000000e+00,-0.000,-0.001
UDPLag,flow_iat_std,7.703,1.720,-5.983,7092443.165,0.000000e+00,0.049,0.009


... (10 outputs left out)


-- SVM ajustado --


,acertos,erros,atributo,SHAP médio nos acertos,SHAP médio nos erros,Δ (erros − acertos),mediana do atributo (acertos),mediana do atributo (erros)
tipo de ataque,,,,,,,,
UDPLag,50,10,down_up_ratio,0.206,-0.549,-0.756,0.0,2.0



-- SVM padrão --


,acertos,erros,atributo,SHAP médio nos acertos,SHAP médio nos erros,Δ (erros − acertos),mediana do atributo (acertos),mediana do atributo (erros)
tipo de ataque,,,,,,,,
Syn,42,18,fwd_packets_s,0.045,-0.340,-0.385,18605.054,2000000.0
UDPLag,45,15,protocol,0.337,-0.396,-0.733,17.000,6.0


## 10. Gráficos e exemplos locais
Importância com IC95%, mapa de calor tipo × atributo, dia do treino × dia do teste, convergência e, nos modelos ajustados, uma linha **certa** e uma **errada** de cada tipo pedido.

In [13]:
# ---------- 10.1 Gráficos (e exemplos locais) ----------
t_sec0 = time.perf_counter(); FIGS = {}
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    for k, M in MODELOS.items():                                                               # importância global (com IC95% bootstrap), os dois dias
        fig, ax = plt.subplots(1, 2, figsize=(13, 5.2), sharex=True)
        for a_, d in zip(ax, DIAS):
            s = GERAL[k][d]; o = np.argsort(-s)[:15][::-1]; lo, hi = BOOT[(k, d)]; a_.barh(range(len(o)), 100 * s[o], xerr=[100 * (s[o] - lo[o]), 100 * (hi[o] - s[o])], color="C0" if d == "treino" else "C3", capsize=2)
            a_.set_yticks(range(len(o))); a_.set_yticklabels([CM[j] for j in o]); a_.set_title(f"{M['rotulo']} | {ROT_DIA[d]} ({len(AMK[k][d]['y']):,} linhas)"); a_.set_xlabel("participação no total de |SHAP| (%; barra = IC95% bootstrap)"); a_.grid(axis="x", alpha=.3)
        fig.suptitle(f"Atributos que mais pesam nas previsões ({'peso igual para cada ' + GRUPO_ROTULO})"); fig.tight_layout(); FIGS[f"importancia_global_{k}.png"] = fig
    for k, M in MODELOS.items():                                                               # mapa de calor: grupo × atributo
        top = np.argsort(-(GERAL[k]["treino"] + GERAL[k]["teste"]))[:20]; fig, ax = plt.subplots(1, 2, figsize=(15, 0.5 * max(len(GRUPOS_DIA["treino"]), len(GRUPOS_DIA["teste"])) + 3.4))
        for a_, d in zip(ax, DIAS):
            m_ = 100 * SHARE[k][d][:, top]; im = a_.imshow(np.nan_to_num(m_), aspect="auto", cmap="viridis"); a_.set_xticks(range(len(top))); a_.set_xticklabels([CM[j] for j in top], rotation=60, ha="right", fontsize=8); a_.set_yticks(range(len(GRUPOS_DIA[d]))); a_.set_yticklabels(GRUPOS_DIA[d])
            a_.set_title(f"{M['rotulo']} | {ROT_DIA[d]}: participação (%) de cada atributo, por {GRUPO_ROTULO}"); fig.colorbar(im, ax=a_, fraction=0.04)
        fig.tight_layout(); FIGS[f"mapa_de_calor_{k}.png"] = fig
    for k, M in MODELOS.items():                                                               # dia do treino × dia do teste
        fig, ax = plt.subplots(figsize=(6.2, 6)); a, b = 100 * GERAL[k]["treino"], 100 * GERAL[k]["teste"]; ax.scatter(a, b, s=18); mx = max(a.max(), b.max()) * 1.05; ax.plot([0, mx], [0, mx], "k--", lw=0.8)
        for j in np.argsort(-np.abs(b - a))[:6]: ax.annotate(CM[j], (a[j], b[j]), fontsize=8, xytext=(3, 3), textcoords="offset points")
        ax.set_xlabel("participação no dia do treino (%)"); ax.set_ylabel("participação no dia do teste (%)"); ax.set_title(f"{M['rotulo']}: o que muda entre os dois dias\n(ρ = {spearman(GERAL[k]['treino'], GERAL[k]['teste']):.2f})"); ax.grid(alpha=.3); fig.tight_layout(); FIGS[f"dia_treino_x_teste_{k}.png"] = fig
    fig, ax = plt.subplots(1, 2, figsize=(11, 4)); 
    for a_, d in zip(ax, DIAS):
        for (mod, dia), g_ in TAB_CONV.groupby(["modelo", "dia"]):
            if dia == ROT_DIA[d]: a_.plot(g_["fração da amostra"], g_["correlação de postos com a amostra inteira"], "o-", label=mod)
        a_.set_title(f"Convergência | {ROT_DIA[d]}"); a_.set_xlabel("fração da amostra"); a_.set_ylabel("correlação de postos com a amostra inteira"); a_.set_ylim(0.5, 1.02); a_.grid(alpha=.3); a_.legend(fontsize=8)
    fig.tight_layout(); FIGS["convergencia.png"] = fig
    # exemplos locais: uma linha CORRETA e uma ERRADA (dia do teste) dos grupos pedidos, nos modelos ajustados
    for k, M in MODELOS.items():
        if not k.endswith("ajustado"): continue
        d = "teste"; A = AMK[k][d]; ok_ = PRED[k][d] == A["y"]; ex_ = [(g, tipo) for g in EXEMPLOS if g in GRUPOS_DIA[d] for tipo in ("acerto", "erro")]; linhas_ = []
        for g, tipo in ex_:
            ix = GIDX[k][d][GRUPOS_DIA[d].index(g)]; cand = ix[ok_[ix]] if tipo == "acerto" else ix[~ok_[ix]]
            if len(cand): linhas_.append((g, tipo, int(cand[0])))
        if linhas_:
            fig, ax = plt.subplots(len(linhas_), 1, figsize=(8, 2.6 * len(linhas_)), squeeze=False)
            for a_, (g, tipo, i) in zip(ax[:, 0], linhas_):
                phi = PHIP[k][d][i]; o = np.argsort(-np.abs(phi))[:8][::-1]; a_.barh(range(len(o)), phi[o], color=["C3" if v > 0 else "C0" for v in phi[o]]); a_.set_yticks(range(len(o))); a_.set_yticklabels([f"{CM[j]} = {A['X'][i, j]:.4g}" for j in o], fontsize=8); a_.axvline(0, color="k", lw=0.6)
                prev = (GRUPOS_DIA[d][int(PRED[k][d][i])] if MODO == "multi" else ("ataque" if PRED[k][d][i] == 1 else "benigno")); a_.set_title(f"{M['rotulo']} | {g} | {tipo} (previsto: {prev})", fontsize=9)
            fig.suptitle("Contribuição de cada atributo (vermelho = aumenta a saída explicada)", fontsize=10); fig.tight_layout(); FIGS[f"exemplos_locais_{k}.png"] = fig
    print(f"✔ {len(FIGS)} gráficos gerados:", ", ".join(FIGS))
except ImportError: print("(matplotlib não está instalado: gráficos não gerados)")
T_SEC["10.1_graficos"] = time.perf_counter() - t_sec0


✔ 15 gráficos gerados: importancia_global_rf_ajustado.png, importancia_global_rf_padrao.png, importancia_global_svm_ajustado.png, importancia_global_svm_padrao.png, mapa_de_calor_rf_ajustado.png, mapa_de_calor_rf_padrao.png, mapa_de_calor_svm_ajustado.png, mapa_de_calor_svm_padrao.png, dia_treino_x_teste_rf_ajustado.png, dia_treino_x_teste_rf_padrao.png, dia_treino_x_teste_svm_ajustado.png, dia_treino_x_teste_svm_padrao.png, convergencia.png, exemplos_locais_rf_ajustado.png, exemplos_locais_svm_ajustado.png


## 11 e 12. Gravação e conferências
Grava os valores SHAP, as tabelas e os gráficos (**só arquivos novos**) e confere: **aditividade**, **amostra estratificada completa**, que os valores gravados reproduzem os calculados e que **nenhum arquivo das pastas dos modelos foi alterado**.

In [14]:
# ---------- 11.1 Gravação ----------
t_sec0 = time.perf_counter(); RES = PASTA_SHAP / "resultados"
for k, M in MODELOS.items():
    for d in DIAS:
        gravar(RES / f"shap_{k}_{d}.npz", lambda p_, k=k, d=d: np.savez_compressed(p_, phi=PHI[k][d], fx=FX[k][d], base=BASE[k][d], y=AMK[k][d]["y"], grupo=AMK[k][d]["grupo"], id=AMK[k][d]["id"], atributos=np.array(CM), grupos=np.array(GRUPOS_DIA[d])))
        cols = {"geral": GERAL[k][d]}; df_s = pd.DataFrame({**{g: SHARE[k][d][i] for i, g in enumerate(GRUPOS_DIA[d])}, "geral (peso igual por grupo)": GERAL[k][d]}, index=CM)
        gravar_csv((100 * df_s).rename_axis("atributo").reset_index(), RES / f"importancia_participacao_pct_{k}_{d}.csv", index=False)
        gravar_csv(pd.DataFrame({g: IMP[k][d][i] for i, g in enumerate(GRUPOS_DIA[d])}, index=CM).rename_axis("atributo").reset_index(), RES / f"importancia_mean_abs_shap_{k}_{d}.csv", index=False)
        gravar_csv(pd.DataFrame({g: DIRE[k][d][i] for i, g in enumerate(GRUPOS_DIA[d])}, index=CM).rename_axis("atributo").reset_index(), RES / f"direcao_corr_valor_shap_{k}_{d}.csv", index=False)
        gravar_csv(TOPG[(k, d)].reset_index(), RES / f"top5_por_grupo_{k}_{d}.csv", index=False)
gravar_csv(tab_amostra("treino", N_POR_GRUPO_RF, N_POR_GRUPO_SVM).rename_axis(GRUPO_ROTULO).reset_index(), RES / "amostra_por_grupo_treino.csv", index=False); gravar_csv(tab_amostra("teste", N_POR_GRUPO_RF, N_POR_GRUPO_SVM).rename_axis(GRUPO_ROTULO).reset_index(), RES / "amostra_por_grupo_teste.csv", index=False)
gravar_csv(TAB_ESTAB, RES / "estabilidade_geral.csv", index=False); gravar_csv(TAB_ESTAB_G, RES / "estabilidade_por_grupo.csv", index=False); gravar_csv(TAB_CONV, RES / "convergencia.csv", index=False)
gravar_csv(TAB_DIAS_RESUMO.reset_index(), RES / "dia_treino_x_teste_correlacao.csv", index=False); gravar_csv(TAB_DIAS_LONGO, RES / "dia_treino_x_teste_atributos_que_mais_mudam.csv", index=False); gravar_csv(TAB_MOD.reset_index(), RES / "rf_x_svm_concordancia.csv", index=False)
if len(TAB_ERROS): gravar_csv(TAB_ERROS, RES / "acertos_x_erros_teste.csv", index=False)
for nome, fig in FIGS.items(): gravar(RES / nome, lambda p_, f=fig: f.savefig(p_, dpi=130, bbox_inches="tight"))
hw = {"so": platform.platform(), "python": sys.version.split()[0], "nucleos_logicos": os.cpu_count()}
if psutil: hw.update({"nucleos_fisicos": psutil.cpu_count(logical=False), "ram_total_gib": psutil.virtual_memory().total / 2 ** 30})
gravar_json({"run_id": RUN_ID, "modo": MODO, "impressao": IMPRESSAO, "modelos": {k: {"rotulo": M["rotulo"], "arquivo": str(M["arquivo"]), "familia": M["familia"], "descricao": M["descricao"]} for k, M in MODELOS.items()}, "atributos": CM, "grupos": GRUPOS_DIA, "n_por_grupo": {"RF": N_POR_GRUPO_RF, "SVM": N_POR_GRUPO_SVM},
             "fundo_svm": {"linhas_por_grupo": N_BG_POR_GRUPO_SVM, "linhas": int(len(FUNDO)), "permutacoes": PERMUTACOES_SVM}, "semente": SEMENTE, "saida_explicada": "multiclasse: a classe REAL da linha (probabilidade no RF; função de decisão um-contra-resto no SVM) | binário: a saída 'ataque' (probabilidade no RF; função de decisão no SVM)",
             "explicadores": {"RF": f"shap.TreeExplainer (exato; shap {shap.__version__})", "SVM": f"shap.PermutationExplainer sobre a função de decisão, com fundo estratificado (shap {shap.__version__})"}, "regras_estabilidade": REGRAS_ESTAB, "versoes": VERSOES, "hardware": hw,
             "paralelismo": {"processos_max": PROCESSOS_MAX, "estatisticas_sessao": {k: (float(v) if isinstance(v, (int, float, np.floating)) else v) for k, v in STATS_RUN.items()}}}, RES / "config_shap_final.json")
T_SEC["11.1_gravacao"] = time.perf_counter() - t_sec0
gravar_csv(pd.DataFrame({"etapa": list(T_SEC), "segundos": [round(v, 1) for v in T_SEC.values()]}), RES / "tempos_etapas.csv", index=False)
print(f"✔ Resultados gravados em {RES}")
# ---------- 12.1 Conferências ----------
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)
verifica("Todos os modelos foram explicados nos dois dias, com valores SHAP finitos", all(np.isfinite(PHI[k][d]).all() for k in MODELOS for d in DIAS), f"({len(MODELOS)} modelos × 2 dias)")
verifica("Aditividade: linha de base + soma dos valores SHAP reproduz a saída do modelo em todas as linhas", max(ADIT.values()) < 1e-4, f"(erro máximo {max(ADIT.values()):.1e})")
verifica("Amostra estratificada: cada grupo tem min(N, linhas disponíveis) linhas, e nenhum grupo ficou de fora", all(int(((AMOSTRA[d]['grupo'] == i) & (AMOSTRA[d]['rank'] < (max(N_POR_GRUPO_RF, N_POR_GRUPO_SVM)))).sum()) == min(max(N_POR_GRUPO_RF, N_POR_GRUPO_SVM), AMOSTRA[d]['disp'][g]) and AMOSTRA[d]['disp'][g] > 0 for d in DIAS for i, g in enumerate(GRUPOS_DIA[d])))
verifica("A amostra do SVM é um subconjunto da do RF (as mesmas linhas, para comparar os dois modelos)", all(set(AMK[a][d]["id"].tolist()) <= set(AMK[b][d]["id"].tolist()) or set(AMK[b][d]["id"].tolist()) <= set(AMK[a][d]["id"].tolist()) for a, b in PARES for d in DIAS) if PARES else True)
verifica("Os valores SHAP gravados reproduzem os calculados (relidos do disco)", all(np.allclose(np.load(RES / f"shap_{k}_{d}.npz")["phi"], PHI[k][d]) for k in MODELOS for d in DIAS))
verifica("As participações somam 100% em cada grupo", all(np.allclose(np.nansum(SHARE[k][d], axis=1), 1.0) for k in MODELOS for d in DIAS) and all(abs(GERAL[k][d].sum() - 1) < 1e-9 for k in MODELOS for d in DIAS))
_depois = {n: v for r_, p_ in PASTAS_MODELOS.items() for n, v in foto(p_, r_).items()}; _mexidos = [n for n, v in SNAP_ANTES.items() if _depois.get(n) != v]
verifica(f"Nenhum dos {len(SNAP_ANTES)} arquivos que já existiam nas pastas dos modelos foi alterado, sobrescrito ou apagado", not _mexidos, f"({_mexidos[:3]})" if _mexidos else "")
verifica("Estabilidade avaliada para todos os modelos e dias", len(TAB_ESTAB) == len(MODELOS) * 2 and TAB_ESTAB["split-half: correlação (média)"].notna().all())
verifica("Arquivos esperados gravados", all((RES / f).exists() for f in ("estabilidade_geral.csv", "convergencia.csv", "dia_treino_x_teste_correlacao.csv", "rf_x_svm_concordancia.csv", "config_shap_final.json", "tempos_etapas.csv", "amostra_por_grupo_teste.csv")) and len(FIGS) > 0)
assert all(ok), "há checagens reprovadas acima"
(PASTA_SHAP / "FINALIZADO").write_text(datetime.now().isoformat(timespec="seconds"), encoding="utf-8")
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas. Resultados em {PASTA_SHAP}"); print(f"\n⏱ Tempo total do notebook (etapas medidas): {fmt_tempo(float(sum(T_SEC.values())))}")
print(f"\nArquivos NOVOS criados nesta execução ({len(CRIADOS)}): {', '.join(CRIADOS[:12])}{' ...' if len(CRIADOS) > 12 else ''}")
print("\nPróximo passo: me mande o texto desta saída (e, se quiser, os arquivos importancia_participacao_pct_*.csv, estabilidade_geral.csv, dia_treino_x_teste_*.csv, acertos_x_erros_teste.csv e os PNG).")


✔ Resultados gravados em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\shap_binario\shap_20261109_173230\resultados
✔ Todos os modelos foram explicados nos dois dias, com valores SHAP finitos (4 modelos × 2 dias)
✔ Aditividade: linha de base + soma dos valores SHAP reproduz a saída do modelo em todas as linhas (erro máximo 6.3e-07)
✔ Amostra estratificada: cada grupo tem min(N, linhas disponíveis) linhas, e nenhum grupo ficou de fora 
✔ A amostra do SVM é um subconjunto da do RF (as mesmas linhas, para comparar os dois modelos) 
✔ Os valores SHAP gravados reproduzem os calculados (relidos do disco) 
✔ As participações somam 100% em cada grupo 
✔ Nenhum dos 778 arquivos que já existiam nas pastas dos modelos foi alterado, sobrescrito ou apagado 
✔ Estabilidade avaliada para todos os modelos e dias 
✔ Arquivos esperados gravados 

✔ 9/9 checagens aprovadas. Resultados em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\shap_binario\shap_20261109_173230
